In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = ROOT / "dataset" / "kosovo_overall_biodiversity.csv"
PROCESSED_DIR = ROOT / "processed"
REPORTS_DIR = ROOT / "reports"
CLEANING_LOG_PATH = PROCESSED_DIR / "cleaning_log.csv"
MISSING_REPORT_PATH = PROCESSED_DIR / "missing_report.csv"
ISSUE_LIST_PATH = REPORTS_DIR / "02_lista_e_problemeve.csv"

PROCESSED_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)


def section(title):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")

# 02 - Integrimi, pastrimi dhe vlerat e zbrazëta

## Shablloni i dokumentimit dhe cleaning log

In [2]:
LOG_COLUMNS = [
    "hapi",
    "input",
    "rreshta_para",
    "kolona_para",
    "rreshta_pas",
    "kolona_pas",
    "arsyeja",
    "output",
    "commit",
]


def init_cleaning_log(path=CLEANING_LOG_PATH):
    """Krijon cleaning_log.csv bosh me kolonat standarde, nëse nuk ekziston."""
    if not path.exists():
        pd.DataFrame(columns=LOG_COLUMNS).to_csv(path, index=False)
    return path


def log_step(
    hapi,
    inputi,
    shape_para,
    shape_pas,
    arsyeja,
    outputi="",
    commit="pending",
    path=CLEANING_LOG_PATH,
    replace_existing=False,
):
    """Shton një rresht në cleaning_log.csv për një hap pastrimi."""
    init_cleaning_log(path)
    row = {
        "hapi": hapi,
        "input": inputi,
        "rreshta_para": shape_para[0],
        "kolona_para": shape_para[1],
        "rreshta_pas": shape_pas[0],
        "kolona_pas": shape_pas[1],
        "arsyeja": arsyeja,
        "output": outputi,
        "commit": commit,
    }
    log = pd.read_csv(path)
    if replace_existing:
        log = log.loc[~(log["hapi"].eq(hapi) & log["input"].eq(inputi))]
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    log.to_csv(path, index=False)
    return log.tail(1)

In [3]:
section("Shablloni i dokumentimit dhe cleaning log")
init_cleaning_log()
print(pd.read_csv(CLEANING_LOG_PATH))


Shablloni i dokumentimit dhe cleaning log
                                  hapi                                    input  rreshta_para  kolona_para  rreshta_pas  kolona_pas  \
0             02_clean:tokens:locality  dataset/kosovo_overall_biodiversity.csv         46831           42        46831          42   
1           02_clean:tokens:recordedBy  dataset/kosovo_overall_biodiversity.csv         46831           42        46831          42   
2         02_clean:tokens:identifiedBy  dataset/kosovo_overall_biodiversity.csv         46831           42        46831          42   
3              02_clean:elevation_zero  dataset/kosovo_overall_biodiversity.csv         46831           42        46831          43   
4      02_clean:drop:iucnRedListStatus  dataset/kosovo_overall_biodiversity.csv         46831           43        46831          42   
..                                 ...                                      ...           ...          ...          ...         ...   
70  02_clean

### Dokumentimi i hapit: Shablloni i dokumentimit dhe cleaning log

1. Input: Struktura e repo-s; folderët notebooks/, processed/, reports/;
   skedari i planifikuar processed/cleaning_log.csv.
2. Çka u bë: U krijua shablloni i dokumentimit dhe funksionet
   init_cleaning_log() / log_step() për regjistrimin e hapave të pastrimit.
3. Rreshta para / rreshta pas: Dataset-i nuk u ndryshua.
   cleaning_log.csv: 0 rreshta, 9 kolona pas inicializimit.
4. Pse u bë: Për të pasur gjurmë të qartë të çdo vendimi dhe për ta bërë
   punën të kontrollueshme nga profesori/grupi.
5. Output: processed/cleaning_log.csv bosh me kolonat standarde.
6. Commit: Add cleaning documentation template and log helper.

## Eksplorimi fillestar i strukturës

In [4]:
section("Eksplorimi fillestar i strukturës")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
rows_before, cols_before = df_raw.shape
print(f"Rreshta: {rows_before:,}")
print(f"Kolona: {cols_before:,}")
print(df_raw.head())

df_raw.info(memory_usage="deep")

dtypes_report = (
    df_raw.dtypes.astype(str)
    .rename("tipi_i_lexuar")
    .reset_index()
    .rename(columns={"index": "kolona"})
)
print("\nTipet e lexuara:")
print(dtypes_report.to_string(index=False))

five_values_per_column = pd.DataFrame({
    "kolona": df_raw.columns,
    "5_vlera_shembull": [
        df_raw[c].dropna().astype(str).drop_duplicates().head(5).tolist()
        for c in df_raw.columns
    ],
    "vlera_unike": [df_raw[c].nunique(dropna=True) for c in df_raw.columns],
})
print("\n5 vlera shembull nga çdo kolonë:")
print(five_values_per_column.to_string(index=False))

issue_rows = []

date_patterns = (
    df_raw["eventDate"]
    .dropna()
    .astype(str)
    .str.replace(r"\d", "9", regex=True)
    .value_counts()
)
if len(date_patterns) > 1:
    issue_rows.append({
        "kategoria": "Format datash",
        "kolona": "eventDate",
        "problemi": f"eventDate paraqitet në {len(date_patterns)} formate/pattern-e të ndryshme.",
        "shembuj": "; ".join(date_patterns.head(5).index.tolist()),
        "veprim_i_sugjeruar": (
            "Parsimi me pd.to_datetime pas ndarjes së intervaleve dhe ruajtja e "
            "precision/date-range flag."
        ),
    })

for c in df_raw.columns:
    values = set(df_raw[c].dropna().astype(str).str.upper().unique())
    if values and values.issubset({"TRUE", "FALSE"}):
        issue_rows.append({
            "kategoria": "Boolean TRUE/FALSE",
            "kolona": c,
            "problemi": "Kolona përmban vlera TRUE/FALSE që duhen standardizuar.",
            "shembuj": ", ".join(sorted(values)),
            "veprim_i_sugjeruar": "Konvertim në boolean ose 0/1 në fazën e pastrimit/binarizimit.",
        })

for c in ["municipality", "district", "protectedArea"]:
    count = df_raw[c].fillna("").astype(str).str.contains("/", regex=False).sum()
    if count > 0:
        examples = (
            df_raw.loc[df_raw[c].fillna("").astype(str).str.contains("/", regex=False), c]
            .dropna()
            .drop_duplicates()
            .head(5)
            .tolist()
        )
        issue_rows.append({
            "kategoria": "Emra dygjuhësh",
            "kolona": c,
            "problemi": f"{count} rreshta përmbajnë emra me ndarës /.",
            "shembuj": "; ".join(examples),
            "veprim_i_sugjeruar": (
                "Ruaj vlerën origjinale dhe, nëse nevojitet, krijo kolonë të "
                "standardizuar për analizë."
            ),
        })

for c in df_raw.columns:
    count = df_raw[c].fillna("").astype(str).str.contains(";", regex=False).sum()
    if count > 0:
        examples = (
            df_raw.loc[df_raw[c].fillna("").astype(str).str.contains(";", regex=False), c]
            .dropna()
            .drop_duplicates()
            .head(3)
            .tolist()
        )
        issue_rows.append({
            "kategoria": "Lista në një kolonë",
            "kolona": c,
            "problemi": f"{count} rreshta kanë disa vlera të ndara me ;.",
            "shembuj": "; ".join(examples),
            "veprim_i_sugjeruar": "Split + explode për raportim ose krijim flamujsh binarë.",
        })

issue_list = pd.DataFrame(issue_rows)
issue_list.to_csv(ISSUE_LIST_PATH, index=False)
print("\nLista e problemeve të vërejtura:")
print(issue_list.to_string(index=False))


Eksplorimi fillestar i strukturës
Rreshta: 46,831
Kolona: 42
       gbifID                     scientificName           species             vernacularName   kingdom        phylum          class           order          family     genus taxonRank   taxonKey  speciesKey  \
0  6500474327     Mantis religiosa (Linne, 1758)  Mantis religiosa             Praying Mantis  Animalia    Arthropoda        Insecta        Mantodea        Mantidae    Mantis   SPECIES  6258028.0   6258028.0   
1  6500462703  Issoria lathonia (Linnaeus, 1758)  Issoria lathonia  Queen of Spain Fritillary  Animalia    Arthropoda        Insecta     Lepidoptera     Nymphalidae   Issoria   SPECIES  1905383.0   1905383.0   
2  6500450749  Podarcis muralis (Laurenti, 1768)  Podarcis muralis         Common Wall Lizard  Animalia      Chordata       Squamata             NaN      Lacertidae  Podarcis   SPECIES  2469188.0   2469188.0   
3  6500428815              Armeria alpina Willd.    Armeria alpina            Alpen-Grasnelke 

<class 'pandas.DataFrame'>
RangeIndex: 46831 entries, 0 to 46830
Data columns (total 42 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   gbifID                         46831 non-null  int64  
 1   scientificName                 46794 non-null  str    
 2   species                        45618 non-null  str    
 3   vernacularName                 42958 non-null  str    
 4   kingdom                        46794 non-null  str    
 5   phylum                         46762 non-null  str    
 6   class                          46443 non-null  str    
 7   order                          46497 non-null  str    
 8   family                         46673 non-null  str    
 9   genus                          46636 non-null  str    
 10  taxonRank                      46794 non-null  str    
 11  taxonKey                       46794 non-null  float64
 12  speciesKey                     45618 non-null  float64
 1


Lista e problemeve të vërejtura:
          kategoria            kolona                                                 problemi                                                                                                                                                                                                                                                                                                                                       shembuj                                                                            veprim_i_sugjeruar
      Format datash         eventDate eventDate paraqitet në 11 formate/pattern-e të ndryshme.                                                                                                                                                                                                                                                             9999-99-99; 9999-99-99T99:99:99; 9999-99-99T99:99; 9999-99-99T99:99:99.999Z; 9999 Parsimi me pd.

### Dokumentimi i hapit: Eksplorimi fillestar i strukturës

1. Input: dataset/kosovo_overall_biodiversity.csv; të gjitha kolonat për
   shape, info, dtypes dhe 5 vlera shembull.
2. Çka u bë: U kontrollua struktura fillestare e dataset-it dhe u krijua
   lista e problemeve të vërejtura: data në formate të ndryshme, boolean
   TRUE/FALSE, emra dygjuhësh dhe lista me ;.
3. Rreshta para / rreshta pas: Dataset-i nuk u ndryshua; df_raw.shape mbetet
   i njëjtë para/pas eksplorimit.
4. Pse u bë: Për të kuptuar strukturën reale para çdo pastrimi dhe për të
   shmangur vendime pa evidencë.
5. Output: reports/02_lista_e_problemeve.csv, dtypes_report,
   five_values_per_column, issue_list.
6. Commit: Add initial structure exploration and issue list.

## Inventari i vlerave të zbrazëta

In [5]:
section("Inventari i vlerave të zbrazëta")
missing_report = pd.DataFrame({
    "kolona": df_raw.columns,
    "mungojne": df_raw.isna().sum().values,
    "perqindja": (df_raw.isna().mean().values * 100).round(2),
    "vlera_te_plotesuara": df_raw.notna().sum().values,
}).sort_values(["perqindja", "kolona"], ascending=[False, True])

missing_report.to_csv(MISSING_REPORT_PATH, index=False)
print(missing_report.to_string(index=False))

print("\nKolonat më problematike për planin e pastrimit:")
print(missing_report[missing_report["mungojne"] > 0].head(15).to_string(index=False))


Inventari i vlerave të zbrazëta
                       kolona  mungojne  perqindja  vlera_te_plotesuara
               iasAppliesFrom     46807      99.95                   24
           establishmentMeans     46591      99.49                  240
                 identifiedBy     39096      83.48                 7735
                    elevation     37011      79.03                 9820
coordinateUncertaintyInMeters     32004      68.34                14827
                protectedArea     25762      55.01                21069
     protectedAreaDesignation     25762      55.01                21069
              individualCount     17697      37.79                29134
                     locality     15251      32.57                31580
               collectionCode     13469      28.76                33362
                catalogNumber     12559      26.82                34272
               vernacularName      3873       8.27                42958
                   recordedBy  

### Dokumentimi i hapit: Inventari i vlerave të zbrazëta

1. Input: dataset/kosovo_overall_biodiversity.csv; të gjitha kolonat.
2. Çka u bë: U llogarit numri dhe përqindja e NaN për çdo kolonë dhe raporti
   u rendit nga kolona më e zbrazët.
3. Rreshta para / rreshta pas: Dataset-i nuk u ndryshua. Raporti ka një
   rresht për çdo kolonë të dataset-it; kolonat e raportit janë kolona,
   mungojne, perqindja, vlera_te_plotesuara.
4. Pse u bë: Për të vendosur strategjinë e trajtimit të vlerave të zbrazëta
   në hapat e pastrimit.
5. Output: processed/missing_report.csv dhe tabela missing_report.
6. Commit: Add missing values inventory.

## Mungesat e fshehura dhe elevation = 0

Leximi i dytë përdor `keep_default_na=False` dhe `na_filter=False`, që `NA`, `null` dhe fushat boshe të mos humben gjatë leximit. Kontrollohen të gjitha kolonat si tekst, me përputhje të plotë pas heqjes së hapësirave dhe normalizimit të shkronjave. Vlera si `N/A` brenda një përshkrimi nuk klasifikohen automatikisht si mungesë.

Raportohen veçmas fushat boshe, vetëm hapësirat, tokenët dhe mungesat që pandas i ka njohur tashmë. Maskat efektive përdorin bashkimin e tyre, pa numërim të dyfishtë. `NE`, `DD`, `False` dhe zero në kolona të tjera nuk janë tokenë mungese.

Burimi gjeografik: [Agjencia e Statistikave të Kosovës, Some Facts on the Environment 2021–2022, f. 10](https://askapi.rks-gov.net/Custom/d9b7ae43-c68e-49a4-b713-5ec33c3a9c25.pdf#page=10), minimumi 270 m. Ky është kontroll plausibiliteti, jo zëvendësim i lartësisë për secilën pikë.

In [6]:
section("Mungesat e fshehura")
from IPython.display import Markdown, display

literal = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False, na_filter=False)
assert literal.shape == df_raw.shape
assert literal["gbifID"].tolist() == df_raw["gbifID"].astype(str).tolist()
token_rows = []
token_mask = pd.DataFrame(False, index=df_raw.index, columns=df_raw.columns)
for column in literal:
    values = literal[column]
    stripped = values.str.strip()
    masks = {"empty_string": values.eq(""), "whitespace_only": values.ne("") & stripped.eq("")}
    masks.update({token: stripped.str.casefold().eq(token) for token in ["unknown", "na", "n/a", "null", "-"]})
    for token, mask in masks.items():
        token_rows.append({"kolona": column, "token": token, "numri": int(mask.sum()),
                           "perqindja": round(mask.mean() * 100, 2),
                           "njohur_si_nan": int((mask & df_raw[column].isna()).sum()),
                           "mungesa_te_reja": int((mask & df_raw[column].notna()).sum())})
        token_mask[column] |= mask
hidden_report = pd.DataFrame(token_rows)
hidden_report.to_csv(REPORTS_DIR / "mungesat_e_fshehura.csv", index=False)

zero_elevation = pd.to_numeric(literal["elevation"], errors="coerce").eq(0)
effective_missing = df_raw.isna() | token_mask
effective_missing["elevation"] |= zero_elevation
effective_report = pd.DataFrame({
    "kolona": df_raw.columns,
    "nan_fillestar": df_raw.isna().sum().values,
    "token_shtese": (token_mask & df_raw.notna()).sum().values,
    "zero_elevation": [int(zero_elevation.sum()) if c == "elevation" else 0 for c in df_raw],
    "mungesa_efektive": effective_missing.sum().values,
    "perqindja_efektive": (effective_missing.mean().values * 100).round(2),
})
effective_report.to_csv(REPORTS_DIR / "raporti_i_mungesave_efektive.csv", index=False)
display(hidden_report[hidden_report["numri"] > 0])
display(effective_report.sort_values("perqindja_efektive", ascending=False))

# Çelësi i burimit mbetet datasetKey; cache-i lokal shërben vetëm për etiketat.
cache_path = PROCESSED_DIR / "gbif_datasets_cache.csv"
source_titles = {}
if cache_path.exists():
    source_cache = pd.read_csv(cache_path)
    assert source_cache["datasetKey"].is_unique
    source_titles = source_cache.set_index("datasetKey")["datasetTitle"].to_dict()

def group_labels(column):
    return literal[column].str.strip().mask(effective_missing[column], "[mungon]")

zero_tables = []
for grouping in ["datasetKey", "basisOfRecord", "district", "municipality", "locality"]:
    frame = pd.DataFrame({"grupi": group_labels(grouping), "zero": zero_elevation,
                          "elevation_present": df_raw["elevation"].notna()})
    table = frame.groupby("grupi", dropna=False).agg(
        n=("zero", "size"), zero_count=("zero", "sum"), elevation_present=("elevation_present", "sum")
    ).reset_index()
    table.insert(0, "grupimi", grouping)
    table["zero_pct_group"] = (table["zero_count"] / table["n"] * 100).round(2)
    table["zero_pct_present"] = (table["zero_count"] / table["elevation_present"].replace(0, np.nan) * 100).round(2)
    table["datasetTitle"] = table["grupi"].map(source_titles) if grouping == "datasetKey" else ""
    zero_tables.append(table)
zero_groups = pd.concat(zero_tables, ignore_index=True)
zero_groups.to_csv(REPORTS_DIR / "lartesia_zero_sipas_grupeve.csv", index=False)
zero_locations = df_raw.loc[zero_elevation, ["gbifID", "datasetKey", "basisOfRecord", "district", "municipality", "locality", "decimalLatitude", "decimalLongitude", "elevation"]]
zero_locations.to_csv(REPORTS_DIR / "regjistrimet_me_lartesi_zero.csv", index=False)
display(zero_groups[(zero_groups["zero_count"] > 0) & (zero_groups["grupimi"] != "locality")])

n_zero = int(zero_elevation.sum())
n_present = int(df_raw["elevation"].notna().sum())
top_source = zero_groups[zero_groups["grupimi"].eq("datasetKey")].sort_values("zero_count", ascending=False).iloc[0]
elevation_decision = f"""# Vendimi për elevation = 0

Ka {n_zero:,} zero ({n_zero / len(df_raw) * 100:.2f}% të rreshtave; {n_zero / n_present * 100:.2f}% të {n_present:,} lartësive të plotësuara).
Burimi kryesor është {top_source['datasetTitle']} (`{top_source['grupi']}`), me {int(top_source['zero_count']):,} zero.
Zerot shfaqen në {zero_locations['district'].nunique()} rajone dhe {zero_locations['municipality'].nunique()} komuna; nuk përfaqësojnë një vend të vetëm të ulët.
Tabelat `lartesia_zero_sipas_grupeve.csv` japin numrin dhe emëruesin sipas burimit, basisOfRecord, rajonit, komunës dhe locality; `regjistrimet_me_lartesi_zero.csv` ruan koordinatat dhe ID-të për auditim.

**Decision + Rationale:** 0 m nuk trajtohet si lartësi reale e sipërfaqes në Kosovë. ASK raporton minimum 270 m në luginën e Drinit të Bardhë ([Some Facts on the Environment 2021–2022, f. 10](https://askapi.rks-gov.net/Custom/d9b7ae43-c68e-49a4-b713-5ec33c3a9c25.pdf#page=10)). Përqendrimi te një burim sugjeron placeholder ose problem eksportimi, por pa metadata të botuesit nuk mund të provohet shkaku për çdo rresht.

Në analizën e mungesave zero përfshihet në maskën efektive; në pastrimin e ardhshëm rekomandohet NaN dhe flamur `elevation_was_zero`, duke ruajtur origjinalin. Nuk zëvendësohet me 270 m ose me mesatare. Mungesa efektive bëhet {effective_missing['elevation'].sum():,} ({effective_missing['elevation'].mean() * 100:.2f}%).

Kufizime: nuk është bërë kontroll GIS/DEM për çdo pikë; lartësitë e tjera të dyshimta (përfshirë maksimumin {df_raw['elevation'].max():g} m) kërkojnë kontroll të veçantë. Referenca kombëtare nuk është prag automatik për të fshirë të gjitha vlerat nën 270 m. Këtu krijohen vetëm raporte, jo dataset i pastruar.
"""
(REPORTS_DIR / "vendimi_per_lartesine_zero.md").write_text(elevation_decision, encoding="utf-8")
display(Markdown(elevation_decision))


Mungesat e fshehura


,kolona,token,numri,perqindja,njohur_si_nan,mungesa_te_reja
7,scientificName,empty_string,37,0.08,37,0
14,species,empty_string,1213,2.59,1213,0
21,vernacularName,empty_string,3873,8.27,3873,0
28,kingdom,empty_string,37,0.08,37,0
35,phylum,empty_string,69,0.15,69,0
42,class,empty_string,388,0.83,388,0
49,order,empty_string,334,0.71,334,0
56,family,empty_string,158,0.34,158,0
63,genus,empty_string,195,0.42,195,0
70,taxonRank,empty_string,37,0.08,37,0


,kolona,nan_fillestar,token_shtese,zero_elevation,mungesa_efektive,perqindja_efektive
41,iasAppliesFrom,46807,0,0,46807,99.95
31,establishmentMeans,46591,0,0,46591,99.49
22,elevation,37011,0,3634,40645,86.79
33,identifiedBy,39096,1,0,39097,83.49
21,coordinateUncertaintyInMeters,32004,0,0,32004,68.34
26,protectedArea,25762,0,0,25762,55.01
27,protectedAreaDesignation,25762,0,0,25762,55.01
30,individualCount,17697,0,0,17697,37.79
23,locality,15251,38,0,15289,32.65
35,collectionCode,13469,0,0,13469,28.76


,grupimi,grupi,n,zero_count,elevation_present,zero_pct_group,zero_pct_present,datasetTitle
95,datasetKey,bc3b0bf0-e245-11dd-ab80-b8a03c50a862,50,10,50,20.00,20.00,World flea collection of Slovenian Museum of Natural History (excluding Slovenia)
110,datasetKey,d92e92e0-8d20-4756-a60b-3746f408c14c,8107,3624,8107,44.70,44.70,Arthropod observations derived from HabiProt Alciphron database in Serbia
125,basisOfRecord,HUMAN_OBSERVATION,42479,3624,8696,8.53,41.67,
131,basisOfRecord,PRESERVED_SPECIMEN,1748,10,1013,0.57,0.99,
132,district,Ferizaj / Uroševac,2593,491,913,18.94,53.78,
133,district,Gjakovë / Đakovica,2697,473,1043,17.54,45.35,
134,district,Gjilan / Gnjilane,4156,9,2285,0.22,0.39,
135,district,Mitrovicë / Kosovska Mitrovica,3196,590,1819,18.46,32.44,
136,district,Pejë / Peć,3187,474,727,14.87,65.20,
137,district,Prishtinë / Priština,25734,678,1151,2.63,58.91,


# Vendimi për elevation = 0

Ka 3,634 zero (7.76% të rreshtave; 37.01% të 9,820 lartësive të plotësuara).
Burimi kryesor është Arthropod observations derived from HabiProt Alciphron database in Serbia (`d92e92e0-8d20-4756-a60b-3746f408c14c`), me 3,624 zero.
Zerot shfaqen në 7 rajone dhe 32 komuna; nuk përfaqësojnë një vend të vetëm të ulët.
Tabelat `lartesia_zero_sipas_grupeve.csv` japin numrin dhe emëruesin sipas burimit, basisOfRecord, rajonit, komunës dhe locality; `regjistrimet_me_lartesi_zero.csv` ruan koordinatat dhe ID-të për auditim.

**Decision + Rationale:** 0 m nuk trajtohet si lartësi reale e sipërfaqes në Kosovë. ASK raporton minimum 270 m në luginën e Drinit të Bardhë ([Some Facts on the Environment 2021–2022, f. 10](https://askapi.rks-gov.net/Custom/d9b7ae43-c68e-49a4-b713-5ec33c3a9c25.pdf#page=10)). Përqendrimi te një burim sugjeron placeholder ose problem eksportimi, por pa metadata të botuesit nuk mund të provohet shkaku për çdo rresht.

Në analizën e mungesave zero përfshihet në maskën efektive; në pastrimin e ardhshëm rekomandohet NaN dhe flamur `elevation_was_zero`, duke ruajtur origjinalin. Nuk zëvendësohet me 270 m ose me mesatare. Mungesa efektive bëhet 40,645 (86.79%).

Kufizime: nuk është bërë kontroll GIS/DEM për çdo pikë; lartësitë e tjera të dyshimta (përfshirë maksimumin 3050 m) kërkojnë kontroll të veçantë. Referenca kombëtare nuk është prag automatik për të fshirë të gjitha vlerat nën 270 m. Këtu krijohen vetëm raporte, jo dataset i pastruar.


### Dokumentimi i hapit: Mungesat e fshehura

1. **Input:** `dataset/kosovo_overall_biodiversity.csv`; inventari NaN dhe raportet e cituara në këtë seksion.
2. **Çka u bë:** U audituan tokenët literalë dhe shpërndarja e elevation = 0.
3. **Rreshta para / pas:** 46,831 / 46,831; **kolona para / pas:** 42 / 42. Maskat diagnostikuese janë të ndara nga dataset-i.
4. **Pse u bë:** Për të mbështetur trajtimin e mungesave me evidencë dhe kuptim semantik.
5. **Output:** `reports/mungesat_e_fshehura.csv`, `raporti_i_mungesave_efektive.csv`, `lartesia_zero_sipas_grupeve.csv`, `regjistrimet_me_lartesi_zero.csv`, `vendimi_per_lartesine_zero.md`.
6. **Commit:** `Add hidden missingness analysis and column strategies` (mesazh i propozuar).
7. **Decision + Rationale:** Zero në elevation përfshihet si mungesë efektive, e arsyetuar në raport; tokenët numërohen pa dyfishuar NaN. Nuk bëhen zëvendësime në df_raw.

## Mekanizmat e mungesës

Përdoren `datasetKey` si burim, `basisOfRecord`, `class` dhe `district` si rajon. `coordinateUncertainty` i detyrës i referohet `coordinateUncertaintyInMeters`. Çdo përqindje ka emërues numrin e rreshtave të grupit, jo të të gjithë dataset-it. Krahasohen NaN fillestar dhe mungesat efektive, përfshirë elevation = 0. Grupet me çelës të zbrazët ruhen si `[mungon]`.

In [7]:
section("Mekanizmat e mungesës")
targets = ["elevation", "individualCount", "coordinateUncertaintyInMeters", "eventDate", "species"]
groupings = ["datasetKey", "basisOfRecord", "class", "district"]
group_tables = []
for grouping in groupings:
    labels = group_labels(grouping)
    for column in targets:
        frame = pd.DataFrame({"grupi": labels, "raw": df_raw[column].isna(), "effective": effective_missing[column]})
        table = frame.groupby("grupi", dropna=False).agg(
            n=("raw", "size"), nan_fillestar=("raw", "sum"), mungesa_efektive=("effective", "sum")
        ).reset_index()
        table.insert(0, "kolona", column)
        table.insert(0, "grupimi", grouping)
        table["pct_nan"] = (table["nan_fillestar"] / table["n"] * 100).round(2)
        table["pct_efektive"] = (table["mungesa_efektive"] / table["n"] * 100).round(2)
        table["grup_i_vogel"] = table["n"] < 30
        table["datasetTitle"] = table["grupi"].map(source_titles) if grouping == "datasetKey" else ""
        group_tables.append(table)
missing_by_group = pd.concat(group_tables, ignore_index=True)
missing_by_group.to_csv(REPORTS_DIR / "mungesat_sipas_grupeve.csv", index=False)
display(missing_by_group[missing_by_group["grupimi"].eq("basisOfRecord")])

interpretations = []
for column in targets:
    eligible = missing_by_group[(missing_by_group["kolona"] == column) & (missing_by_group["n"] >= 30)]
    descriptions = []
    for grouping in groupings:
        subset = eligible[eligible["grupimi"] == grouping]
        if len(subset) >= 2:
            low = subset.loc[subset["pct_efektive"].idxmin()]
            high = subset.loc[subset["pct_efektive"].idxmax()]
            descriptions.append(f"{grouping}: {low['pct_efektive']:.2f}% ({low['grupi']}, n={low['n']}) deri {high['pct_efektive']:.2f}% ({high['grupi']}, n={high['n']})")
    interpretations.append(f"- **{column}**: " + "; ".join(descriptions) + ".")
mechanisms_report = """# Mekanizmat e mungesës

Përqindjet bazohen në maskat efektive të seksionit paraprak. CSV-ja ruan edhe numërimet dhe përqindjet origjinale NaN. Grupimet janë margjinale, jo kombinime të katër faktorëve. Të gjitha grupet ruhen në CSV; më poshtë krahasohen vetëm grupet me n >= 30 për të mos mbështetur interpretimin te grupe tepër të vogla. Ky kufi është vetëm për përmbledhjen, jo prag imputimi apo test statistikor.

""" + "\n\n".join(interpretations) + """

**Decision + Rationale:** Përqindjet ndryshojnë sipas karakteristikave të vëzhguara. The missingness pattern appears consistent with MAR conditional on observed source and record characteristics, but MAR is not established by these descriptive comparisons. Prandaj MCAR nuk merret si supozim pune dhe nuk përdoret imputim i përgjithshëm me mesatare ose modë.

Kufizime: tabelat nuk provojnë MAR dhe nuk përjashtojnë MNAR; vlerat e pavëzhguara nuk mund të testohen nga ky CSV. Burimi, klasa, rajoni dhe basisOfRecord janë të ndërvarura; nuk interpretohen si shkaqe të pavarura. Vëzhgimet nuk janë mostër e rastësishme dhe mund të kenë varësi hapësinore, kohore ose dublikime. Për species mungesa mund të lidhet me rangun taksonomik; për individualCount me mënyrën e raportimit; për elevation me konventat e burimit. Këto janë shpjegime të mundshme, jo mekanizma të provuar. Datat e pjesshme të plotësuara nuk janë NaN: precision duhet ruajtur veçmas në pastrim. Nevojiten metadata të burimeve dhe analizë ndjeshmërie para imputimit modelor.
"""
(REPORTS_DIR / "mekanizmat_e_mungeses.md").write_text(mechanisms_report, encoding="utf-8")
display(Markdown(mechanisms_report))


Mekanizmat e mungesës


,grupimi,kolona,grupi,n,nan_fillestar,mungesa_efektive,pct_nan,pct_efektive,grup_i_vogel,datasetTitle
625,basisOfRecord,elevation,HUMAN_OBSERVATION,42479,33783,37407,79.53,88.06,False,
626,basisOfRecord,elevation,MACHINE_OBSERVATION,376,376,376,100.00,100.00,False,
627,basisOfRecord,elevation,MATERIAL_CITATION,317,292,292,92.11,92.11,False,
628,basisOfRecord,elevation,MATERIAL_SAMPLE,641,605,605,94.38,94.38,False,
629,basisOfRecord,elevation,OBSERVATION,1,0,0,0.00,0.00,True,
630,basisOfRecord,elevation,OCCURRENCE,1269,1220,1220,96.14,96.14,False,
631,basisOfRecord,elevation,PRESERVED_SPECIMEN,1748,735,745,42.05,42.62,False,
632,basisOfRecord,individualCount,HUMAN_OBSERVATION,42479,13828,13828,32.55,32.55,False,
633,basisOfRecord,individualCount,MACHINE_OBSERVATION,376,363,363,96.54,96.54,False,
634,basisOfRecord,individualCount,MATERIAL_CITATION,317,100,100,31.55,31.55,False,


# Mekanizmat e mungesës

Përqindjet bazohen në maskat efektive të seksionit paraprak. CSV-ja ruan edhe numërimet dhe përqindjet origjinale NaN. Grupimet janë margjinale, jo kombinime të katër faktorëve. Të gjitha grupet ruhen në CSV; më poshtë krahasohen vetëm grupet me n >= 30 për të mos mbështetur interpretimin te grupe tepër të vogla. Ky kufi është vetëm për përmbledhjen, jo prag imputimi apo test statistikor.

- **elevation**: datasetKey: 0.00% (03269e13-84ae-430f-990e-f11069413e36, n=52) deri 100.00% (07044577-bd82-4089-9f3a-f4a9d2170b2e, n=386); basisOfRecord: 42.62% (PRESERVED_SPECIMEN, n=1748) deri 100.00% (MACHINE_OBSERVATION, n=376); class: 3.76% (Lecanoromycetes, n=612) deri 100.00% (Amphibia, n=109); district: 45.24% (Gjilan / Gnjilane, n=4156) deri 98.16% (Prishtinë / Priština, n=25734).

- **individualCount**: datasetKey: 0.00% (03269e13-84ae-430f-990e-f11069413e36, n=52) deri 100.00% (040c5662-da76-4782-a48e-cdea1892d14c, n=341); basisOfRecord: 31.55% (MATERIAL_CITATION, n=317) deri 96.54% (MACHINE_OBSERVATION, n=376); class: 10.63% (Aves, n=28785) deri 100.00% (Bivalvia, n=40); district: 15.00% (Prishtinë / Priština, n=25734) deri 71.87% (Prizren, n=5268).

- **coordinateUncertaintyInMeters**: datasetKey: 0.00% (6ac3f774-d9fb-4796-b3e9-92bf6c81c084, n=721) deri 100.00% (03269e13-84ae-430f-990e-f11069413e36, n=52); basisOfRecord: 66.57% (HUMAN_OBSERVATION, n=42479) deri 97.79% (MATERIAL_CITATION, n=317); class: 11.36% (Mammalia, n=273) deri 100.00% (Clitellata, n=43); district: 30.38% (Mitrovicë / Kosovska Mitrovica, n=3196) deri 87.13% (Prishtinë / Priština, n=25734).

- **eventDate**: datasetKey: 0.00% (03269e13-84ae-430f-990e-f11069413e36, n=52) deri 100.00% (69bbc9fb-a432-4945-bb4f-654bff3bc2de, n=46); basisOfRecord: 0.00% (MACHINE_OBSERVATION, n=376) deri 52.01% (OCCURRENCE, n=1269); class: 0.00% (Agaricomycetes, n=187) deri 95.00% (Bivalvia, n=40); district: 0.85% (Prishtinë / Priština, n=25734) deri 16.49% (Mitrovicë / Kosovska Mitrovica, n=3196).

- **species**: datasetKey: 0.00% (03269e13-84ae-430f-990e-f11069413e36, n=52) deri 94.51% (bd14614c-4d72-4763-8710-1cea514311cc, n=182); basisOfRecord: 0.00% (MACHINE_OBSERVATION, n=376) deri 55.23% (MATERIAL_SAMPLE, n=641); class: 0.00% (Amphibia, n=109) deri 62.50% (Bivalvia, n=40); district: 1.44% (Gjilan / Gnjilane, n=4156) deri 7.66% (Pejë / Peć, n=3187).

**Decision + Rationale:** Përqindjet ndryshojnë sipas karakteristikave të vëzhguara. The missingness pattern appears consistent with MAR conditional on observed source and record characteristics, but MAR is not established by these descriptive comparisons. Prandaj MCAR nuk merret si supozim pune dhe nuk përdoret imputim i përgjithshëm me mesatare ose modë.

Kufizime: tabelat nuk provojnë MAR dhe nuk përjashtojnë MNAR; vlerat e pavëzhguara nuk mund të testohen nga ky CSV. Burimi, klasa, rajoni dhe basisOfRecord janë të ndërvarura; nuk interpretohen si shkaqe të pavarura. Vëzhgimet nuk janë mostër e rastësishme dhe mund të kenë varësi hapësinore, kohore ose dublikime. Për species mungesa mund të lidhet me rangun taksonomik; për individualCount me mënyrën e raportimit; për elevation me konventat e burimit. Këto janë shpjegime të mundshme, jo mekanizma të provuar. Datat e pjesshme të plotësuara nuk janë NaN: precision duhet ruajtur veçmas në pastrim. Nevojiten metadata të burimeve dhe analizë ndjeshmërie para imputimit modelor.


### Dokumentimi i hapit: Mekanizmat e mungesës

1. **Input:** `dataset/kosovo_overall_biodiversity.csv`; inventari NaN dhe raportet e cituara në këtë seksion.
2. **Çka u bë:** U llogaritën numërimet, emëruesit dhe përqindjet për pesë kolona sipas katër grupimeve.
3. **Rreshta para / pas:** 46,831 / 46,831; **kolona para / pas:** 42 / 42. Maskat diagnostikuese janë të ndara nga dataset-i.
4. **Pse u bë:** Për të mbështetur trajtimin e mungesave me evidencë dhe kuptim semantik.
5. **Output:** `reports/mungesat_sipas_grupeve.csv`, `reports/mekanizmat_e_mungeses.md`.
6. **Commit:** `Add hidden missingness analysis and column strategies` (mesazh i propozuar).
7. **Decision + Rationale:** MAR është vetëm supozim i mundshëm i kushtëzuar; krahasimet përshkruese nuk e provojnë dhe nuk përjashtojnë MNAR.

## Strategjia për vlerat e zbrazëta

Rëndësia analitike merret nga `reports/column_importance.md`; vendimi për redundancën nga `reports/redundancy.md`. Strategjia është plan për fazën e pastrimit: `delete` nënkupton heqje vetëm nga pamja analitike e përcaktuar, jo nga origjinali. `impute` më poshtë nënkupton rikuperim determinist të verifikuar, jo krijim vlerash me mesatare/modë. Nuk përdoret prag i përgjithshëm i mungesës.

In [8]:
section("Strategjia për vlerat e zbrazëta")
import re

importance_text = (REPORTS_DIR / "column_importance.md").read_text(encoding="utf-8")
importance = dict(re.findall(r"^\| `([^`]+)` \| [^|]+ \| ([^|]+) \|$", importance_text, re.MULTILINE))
assert set(df_raw.columns) <= importance.keys(), "Mungon vlerësimi i rëndësisë për disa kolona."
strategies = {}

def strategy(columns, recovery, decision, reason):
    for column in columns.split():
        strategies[column] = (recovery, decision, reason)

strategy("gbifID datasetKey", "Vetëm nga burimi origjinal; jo gjenerim ID-sh.", "keep", "Identifikues për gjurmueshmëri dhe grupim; ruhet edhe kur rëndësia direkte analitike është e ulët.")
strategy("scientificName taxonRank", "Validim me taksonominë e burimit.", "keep + flag", "Ruhen emri dhe rangu real; mungesa nuk zëvendësohet me taksonin më të shpeshtë.")
strategy("species speciesKey", "Vetëm lidhje e verifikuar me speciesKey/taksonominë.", "keep + flag", "Mos shpik species nga gjinia. Përjashto vetëm nga analiza në nivel species, jo nga dataset-i i përgjithshëm.")
strategy("kingdom phylum class order family genus taxonKey", "Hierarki taksonomike e verifikuar nga çelësi GBIF.", "keep + flag", "Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet.")
strategy("vernacularName", "Fjalor taksonomik me gjuhë të përcaktuar.", "keep", "Emri i zakonshëm është opsional; mungesa nuk pengon identifikimin shkencor.")
strategy("iucnRedListCategory", "Vetëm vlerësim IUCN i verifikuar për taksonin dhe versionin.", "keep + flag", "NE dhe DD janë kategori reale, jo NaN dhe jo LC; mos imputo status rreziku.")
strategy("iucnRedListStatus", "Rindërtohet nga harta e kodeve të IUCN.", "delete (pamja analitike)", "Redundant sipas reports/redundancy.md; jo vendim i bazuar në përqindjen e mungesës.")
strategy("eventDate", "Komponentë year/month/day vetëm nëse janë të vlefshëm e konsistentë.", "keep + impute + flag", "Ruaj intervalet dhe precision; mos krijo muaj/ditë nga data vetëm me vit. Rikuperimi i ardhshëm shënohet.")
strategy("year month day", "Nga eventDate vetëm kur komponenti është shprehimisht i pranishëm dhe i vlefshëm.", "keep + impute + flag", "Mos përdor 1 janar si datë reale; rreshtat pa komponentin e nevojshëm përjashtohen vetëm nga analiza kohore përkatëse.")
strategy("decimalLatitude decimalLongitude", "Vetëm gjeoreferencim i dokumentuar.", "keep + flag", "Pa mesatare koordinatash; ruaj pikën dhe verifiko vlefshmërinë/hartën veçmas.")
strategy("coordinateUncertaintyInMeters", "Metadata e gjeoreferencimit nga burimi.", "keep + flag", "Mungesa nuk është pasiguri zero. Analizat hapësinore duhet të raportojnë filtrat e cilësisë.")
strategy("elevation", "DEM i verifikuar nga koordinatat, me rezolucion dhe proveniencë; ende jo zbatuar.", "keep + flag", "Zero trajtohet si NaN në kopjen e ardhshme me elevation_was_zero; pa medianë globale. Përdorim vetëm pas kontrollit të plausibilitetit.")
strategy("locality", "Burimi origjinal; komuna nuk zëvendëson lokalitetin.", "keep + flag", "unknown dhe tokenët bosh normalizohen në kopjen e pastruar; mos shpik lokalitet specifik.")
strategy("municipality district", "Spatial join me kufij administrativë të verifikuar nëse nevojitet.", "keep", "Kolona të plota për analizë rajonale; harmonizimi i emrave është hap i veçantë.")
strategy("protectedArea protectedAreaDesignation", "Spatial join me kufijtë dhe versionin e zonave të mbrojtura.", "keep + flag", "Raportet ekzistuese sugjerojnë jashtë zonës, por mungesa vetëm nuk provon Not protected. Mbaj unknown derisa mbulimi hapësinor të verifikohet.")
strategy("strictlyProtected", "Metadata/hartë e mbrojtjes.", "keep", "False është vlerë e vlefshme; mungesa eventuale nuk zëvendësohet automatikisht me False.")
strategy("basisOfRecord", "Metadata e burimit.", "keep", "Përcakton kuptimin e regjistrimit dhe dallimet e mungesës; nuk imputohet me modë.")
strategy("individualCount", "Vetëm numër i raportuar nga burimi.", "keep + flag", "Mungesa nuk është 0 ose 1. Mos ngatërro numrin e regjistrimeve me numrin e individëve; pa imputim global.")
strategy("establishmentMeans", "Informacion ekologjik i verifikuar.", "delete (pamja kryesore)", "Pothuajse bosh dhe jashtë pyetjeve kryesore sipas rëndësisë; ruaj origjinalin për analizë të specializuar.")
strategy("recordedBy identifiedBy", "Metadata e regjistrimit.", "keep + flag", "unknown normalizohet; mos imputo identitetin me emrin më të shpeshtë.")
strategy("institutionCode collectionCode catalogNumber", "Metadata e institucionit/koleksionit për regjistrimin.", "keep", "Fusha provenience opsionale; datasetKey nuk përcakton gjithmonë një institucion, prandaj jo mbushje e verbër.")
strategy("license", "Licenca e regjistrimit/burimit.", "keep", "Duhet për përdorim dhe atribuim pavarësisht rëndësisë së ulët analitike.")
strategy("issue", "Flag-et e eksportit GBIF.", "keep + flag", "Bosh mund të nënkuptojë pa flag të raportuar, jo cilësi të garantuar; mos imputo problem.")
strategy("iasUnionConcern", "Lista dhe metadata për versionin përkatës.", "keep", "False është vlerë e vlefshme; nuk tregon domosdoshmërisht plotësinë e njohurive ekologjike.")
strategy("iasAppliesFrom", "Data e listimit vetëm kur zbatohet iasUnionConcern.", "keep + flag", "Mungesa mund të jetë strukturore për False. Ndaj not applicable nga unknown kur True; mos hiq vetëm për shkak të 99.95% mungesës.")
assert set(strategies) == set(df_raw.columns)

def markdown_table(frame):
    def escape(value):
        return str(value).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(map(escape, frame.columns)) + " |", "| " + " | ".join(["---"] * len(frame.columns)) + " |"]
    lines.extend("| " + " | ".join(map(escape, row)) + " |" for row in frame.itertuples(index=False, name=None))
    return "\n".join(lines)

strategy_rows = []
for column in df_raw:
    recovery, decision, reason = strategies[column]
    strategy_rows.append({"Column": column, "Missing %": f"{effective_missing[column].mean() * 100:.2f}",
                          "Rëndësia": importance[column].strip(), "Rikuperimi": recovery,
                          "Vendimi": decision, "Arsyetimi": reason})
strategy_table = pd.DataFrame(strategy_rows)
ias_check = pd.crosstab(literal["iasUnionConcern"], effective_missing["iasAppliesFrom"], dropna=False)
strategy_report = """# Strategjia për vlerat e zbrazëta

Input: dataset-i origjinal, `raporti_i_mungesave_efektive.csv`, `mungesat_sipas_grupeve.csv`, `mekanizmat_e_mungeses.md`, `column_importance.md` dhe `redundancy.md`.

**Missing %** është përqindja efektive: bashkimi i NaN, tokenëve literalë të mungesës dhe elevation = 0. Inventari fillestar mbetet në `processed/missing_report.csv`; numërimet e krahasueshme janë në `reports/raporti_i_mungesave_efektive.csv`. NE/DD janë kategori IUCN; mungesat potencialisht strukturore ende numërohen dhe interpretohen veçmas.

**Decision + Rationale:** Nuk ka prag fiks. Vendimet mbështeten në rëndësi, kuptim, mundësi rikuperimi dhe modelin e mungesës. Delete vlen vetëm për pamjen e specifikuar; keep ruan vlerën dhe NaN; flag shton në fazën e zbatimit një tregues mungese; impute lejohet vetëm për rikuperim determinist të verifikuar. Asnjë vendim në tabelë nuk është zbatuar mbi dataset-in. Nuk ka fshirje globale rreshtash për species/year. Përqindjet e larta nuk justifikojnë vetë heqjen.

""" + markdown_table(strategy_table) + """

## Kontrolli i mungesës strukturore

iasAppliesFrom sipas iasUnionConcern (False/True në kolonat më poshtë tregojnë nëse data mungon):

""" + markdown_table(ias_check.reset_index()) + """

Për protectedArea raporti i mëparshëm e interpreton boshllëkun si jashtë zonave të mbrojtura. Këtu kjo mbetet hipotezë deri në verifikimin e mbulimit dhe versionit të spatial join; nuk caktohet automatikisht Not protected.

## Dokumentimi i detyrës

- **Input:** dataset/kosovo_overall_biodiversity.csv dhe raportet e mësipërme.
- **Çka u bë:** U vendos strategji për të 42 kolonat me rëndësi, rikuperim dhe arsyetim.
- **Rreshta para / pas:** 46,831 / 46,831; **kolona para / pas:** 42 / 42.
- **Pse u bë:** Për të shmangur humbje informacioni dhe imputime pa mbështetje semantike.
- **Output:** reports/strategjia_e_mungesave.md, 42 vendime.
- **Commit:** Add hidden missingness analysis and column strategies (mesazh i propozuar).

Këto janë hapa diagnostikimi dhe planifikimi; cleaning_log.csv ruhet për transformimet reale. Zbatimi i vendimeve duhet të regjistrojë ndryshimet me log_step().
"""
(REPORTS_DIR / "strategjia_e_mungesave.md").write_text(strategy_report, encoding="utf-8")
display(Markdown(strategy_report))


Strategjia për vlerat e zbrazëta


# Strategjia për vlerat e zbrazëta

Input: dataset-i origjinal, `raporti_i_mungesave_efektive.csv`, `mungesat_sipas_grupeve.csv`, `mekanizmat_e_mungeses.md`, `column_importance.md` dhe `redundancy.md`.

**Missing %** është përqindja efektive: bashkimi i NaN, tokenëve literalë të mungesës dhe elevation = 0. Inventari fillestar mbetet në `processed/missing_report.csv`; numërimet e krahasueshme janë në `reports/raporti_i_mungesave_efektive.csv`. NE/DD janë kategori IUCN; mungesat potencialisht strukturore ende numërohen dhe interpretohen veçmas.

**Decision + Rationale:** Nuk ka prag fiks. Vendimet mbështeten në rëndësi, kuptim, mundësi rikuperimi dhe modelin e mungesës. Delete vlen vetëm për pamjen e specifikuar; keep ruan vlerën dhe NaN; flag shton në fazën e zbatimit një tregues mungese; impute lejohet vetëm për rikuperim determinist të verifikuar. Asnjë vendim në tabelë nuk është zbatuar mbi dataset-in. Nuk ka fshirje globale rreshtash për species/year. Përqindjet e larta nuk justifikojnë vetë heqjen.

| Column | Missing % | Rëndësia | Rikuperimi | Vendimi | Arsyetimi |
| --- | --- | --- | --- | --- | --- |
| gbifID | 0.00 | e ulët | Vetëm nga burimi origjinal; jo gjenerim ID-sh. | keep | Identifikues për gjurmueshmëri dhe grupim; ruhet edhe kur rëndësia direkte analitike është e ulët. |
| scientificName | 0.08 | e lartë | Validim me taksonominë e burimit. | keep + flag | Ruhen emri dhe rangu real; mungesa nuk zëvendësohet me taksonin më të shpeshtë. |
| species | 2.59 | e lartë | Vetëm lidhje e verifikuar me speciesKey/taksonominë. | keep + flag | Mos shpik species nga gjinia. Përjashto vetëm nga analiza në nivel species, jo nga dataset-i i përgjithshëm. |
| vernacularName | 8.27 | e ulët | Fjalor taksonomik me gjuhë të përcaktuar. | keep | Emri i zakonshëm është opsional; mungesa nuk pengon identifikimin shkencor. |
| kingdom | 0.08 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| phylum | 0.15 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| class | 0.83 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| order | 0.71 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| family | 0.34 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| genus | 0.42 | e lartë | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| taxonRank | 0.08 | e mesme | Validim me taksonominë e burimit. | keep + flag | Ruhen emri dhe rangu real; mungesa nuk zëvendësohet me taksonin më të shpeshtë. |
| taxonKey | 0.08 | e mesme | Hierarki taksonomike e verifikuar nga çelësi GBIF. | keep + flag | Ruaj nivelin e identifikimit; rikuperim vetëm kur lidhja është e qartë dhe dokumentohet. |
| speciesKey | 2.59 | e mesme | Vetëm lidhje e verifikuar me speciesKey/taksonominë. | keep + flag | Mos shpik species nga gjinia. Përjashto vetëm nga analiza në nivel species, jo nga dataset-i i përgjithshëm. |
| iucnRedListCategory | 2.59 | e lartë | Vetëm vlerësim IUCN i verifikuar për taksonin dhe versionin. | keep + flag | NE dhe DD janë kategori reale, jo NaN dhe jo LC; mos imputo status rreziku. |
| iucnRedListStatus | 2.59 | e ulët | Rindërtohet nga harta e kodeve të IUCN. | delete (pamja analitike) | Redundant sipas reports/redundancy.md; jo vendim i bazuar në përqindjen e mungesës. |
| eventDate | 4.68 | e lartë | Komponentë year/month/day vetëm nëse janë të vlefshëm e konsistentë. | keep + impute + flag | Ruaj intervalet dhe precision; mos krijo muaj/ditë nga data vetëm me vit. Rikuperimi i ardhshëm shënohet. |
| year | 4.84 | e lartë | Nga eventDate vetëm kur komponenti është shprehimisht i pranishëm dhe i vlefshëm. | keep + impute + flag | Mos përdor 1 janar si datë reale; rreshtat pa komponentin e nevojshëm përjashtohen vetëm nga analiza kohore përkatëse. |
| month | 6.12 | e lartë | Nga eventDate vetëm kur komponenti është shprehimisht i pranishëm dhe i vlefshëm. | keep + impute + flag | Mos përdor 1 janar si datë reale; rreshtat pa komponentin e nevojshëm përjashtohen vetëm nga analiza kohore përkatëse. |
| day | 6.87 | e mesme | Nga eventDate vetëm kur komponenti është shprehimisht i pranishëm dhe i vlefshëm. | keep + impute + flag | Mos përdor 1 janar si datë reale; rreshtat pa komponentin e nevojshëm përjashtohen vetëm nga analiza kohore përkatëse. |
| decimalLatitude | 0.00 | e mesme | Vetëm gjeoreferencim i dokumentuar. | keep + flag | Pa mesatare koordinatash; ruaj pikën dhe verifiko vlefshmërinë/hartën veçmas. |
| decimalLongitude | 0.00 | e mesme | Vetëm gjeoreferencim i dokumentuar. | keep + flag | Pa mesatare koordinatash; ruaj pikën dhe verifiko vlefshmërinë/hartën veçmas. |
| coordinateUncertaintyInMeters | 68.34 | e lartë | Metadata e gjeoreferencimit nga burimi. | keep + flag | Mungesa nuk është pasiguri zero. Analizat hapësinore duhet të raportojnë filtrat e cilësisë. |
| elevation | 86.79 | e ulët | DEM i verifikuar nga koordinatat, me rezolucion dhe proveniencë; ende jo zbatuar. | keep + flag | Zero trajtohet si NaN në kopjen e ardhshme me elevation_was_zero; pa medianë globale. Përdorim vetëm pas kontrollit të plausibilitetit. |
| locality | 32.65 | e mesme | Burimi origjinal; komuna nuk zëvendëson lokalitetin. | keep + flag | unknown dhe tokenët bosh normalizohen në kopjen e pastruar; mos shpik lokalitet specifik. |
| municipality | 0.00 | e lartë | Spatial join me kufij administrativë të verifikuar nëse nevojitet. | keep | Kolona të plota për analizë rajonale; harmonizimi i emrave është hap i veçantë. |
| district | 0.00 | e mesme | Spatial join me kufij administrativë të verifikuar nëse nevojitet. | keep | Kolona të plota për analizë rajonale; harmonizimi i emrave është hap i veçantë. |
| protectedArea | 55.01 | e lartë | Spatial join me kufijtë dhe versionin e zonave të mbrojtura. | keep + flag | Raportet ekzistuese sugjerojnë jashtë zonës, por mungesa vetëm nuk provon Not protected. Mbaj unknown derisa mbulimi hapësinor të verifikohet. |
| protectedAreaDesignation | 55.01 | e lartë | Spatial join me kufijtë dhe versionin e zonave të mbrojtura. | keep + flag | Raportet ekzistuese sugjerojnë jashtë zonës, por mungesa vetëm nuk provon Not protected. Mbaj unknown derisa mbulimi hapësinor të verifikohet. |
| strictlyProtected | 0.00 | e mesme | Metadata/hartë e mbrojtjes. | keep | False është vlerë e vlefshme; mungesa eventuale nuk zëvendësohet automatikisht me False. |
| basisOfRecord | 0.00 | e lartë | Metadata e burimit. | keep | Përcakton kuptimin e regjistrimit dhe dallimet e mungesës; nuk imputohet me modë. |
| individualCount | 37.79 | e ulët | Vetëm numër i raportuar nga burimi. | keep + flag | Mungesa nuk është 0 ose 1. Mos ngatërro numrin e regjistrimeve me numrin e individëve; pa imputim global. |
| establishmentMeans | 99.49 | e ulët | Informacion ekologjik i verifikuar. | delete (pamja kryesore) | Pothuajse bosh dhe jashtë pyetjeve kryesore sipas rëndësisë; ruaj origjinalin për analizë të specializuar. |
| recordedBy | 8.23 | e ulët | Metadata e regjistrimit. | keep + flag | unknown normalizohet; mos imputo identitetin me emrin më të shpeshtë. |
| identifiedBy | 83.49 | e ulët | Metadata e regjistrimit. | keep + flag | unknown normalizohet; mos imputo identitetin me emrin më të shpeshtë. |
| institutionCode | 7.92 | e mesme | Metadata e institucionit/koleksionit për regjistrimin. | keep | Fusha provenience opsionale; datasetKey nuk përcakton gjithmonë një institucion, prandaj jo mbushje e verbër. |
| collectionCode | 28.76 | e ulët | Metadata e institucionit/koleksionit për regjistrimin. | keep | Fusha provenience opsionale; datasetKey nuk përcakton gjithmonë një institucion, prandaj jo mbushje e verbër. |
| catalogNumber | 26.82 | e ulët | Metadata e institucionit/koleksionit për regjistrimin. | keep | Fusha provenience opsionale; datasetKey nuk përcakton gjithmonë një institucion, prandaj jo mbushje e verbër. |
| datasetKey | 0.00 | e mesme | Vetëm nga burimi origjinal; jo gjenerim ID-sh. | keep | Identifikues për gjurmueshmëri dhe grupim; ruhet edhe kur rëndësia direkte analitike është e ulët. |
| license | 0.00 | e ulët | Licenca e regjistrimit/burimit. | keep | Duhet për përdorim dhe atribuim pavarësisht rëndësisë së ulët analitike. |
| issue | 3.04 | e lartë | Flag-et e eksportit GBIF. | keep + flag | Bosh mund të nënkuptojë pa flag të raportuar, jo cilësi të garantuar; mos imputo problem. |
| iasUnionConcern | 0.00 | e ulët | Lista dhe metadata për versionin përkatës. | keep | False është vlerë e vlefshme; nuk tregon domosdoshmërisht plotësinë e njohurive ekologjike. |
| iasAppliesFrom | 99.95 | e ulët | Data e listimit vetëm kur zbatohet iasUnionConcern. | keep + flag | Mungesa mund të jetë strukturore për False. Ndaj not applicable nga unknown kur True; mos hiq vetëm për shkak të 99.95% mungesës. |

## Kontrolli i mungesës strukturore

iasAppliesFrom sipas iasUnionConcern (False/True në kolonat më poshtë tregojnë nëse data mungon):

| iasUnionConcern | False | True |
| --- | --- | --- |
| FALSE | 0 | 46807 |
| TRUE | 24 | 0 |

Për protectedArea raporti i mëparshëm e interpreton boshllëkun si jashtë zonave të mbrojtura. Këtu kjo mbetet hipotezë deri në verifikimin e mbulimit dhe versionit të spatial join; nuk caktohet automatikisht Not protected.

## Dokumentimi i detyrës

- **Input:** dataset/kosovo_overall_biodiversity.csv dhe raportet e mësipërme.
- **Çka u bë:** U vendos strategji për të 42 kolonat me rëndësi, rikuperim dhe arsyetim.
- **Rreshta para / pas:** 46,831 / 46,831; **kolona para / pas:** 42 / 42.
- **Pse u bë:** Për të shmangur humbje informacioni dhe imputime pa mbështetje semantike.
- **Output:** reports/strategjia_e_mungesave.md, 42 vendime.
- **Commit:** Add hidden missingness analysis and column strategies (mesazh i propozuar).

Këto janë hapa diagnostikimi dhe planifikimi; cleaning_log.csv ruhet për transformimet reale. Zbatimi i vendimeve duhet të regjistrojë ndryshimet me log_step().


### Dokumentimi i hapit: Strategjia e mungesave

1. **Input:** `dataset/kosovo_overall_biodiversity.csv`; inventari NaN dhe raportet e cituara në këtë seksion.
2. **Çka u bë:** U përgatit vendimi për secilën nga 42 kolonat, duke përdorur rëndësinë analitike të Jetës.
3. **Rreshta para / pas:** 46,831 / 46,831; **kolona para / pas:** 42 / 42. Maskat diagnostikuese janë të ndara nga dataset-i.
4. **Pse u bë:** Për të mbështetur trajtimin e mungesave me evidencë dhe kuptim semantik.
5. **Output:** `reports/strategjia_e_mungesave.md` (42 rreshta strategjie).
6. **Commit:** `Add hidden missingness analysis and column strategies` (mesazh i propozuar).
7. **Decision + Rationale:** Pa prag të përgjithshëm; rikuperimi determinist është i kushtëzuar nga verifikimi. Vendimet nuk janë zbatuar ende si transformime.

## Zbatimi i vendimeve për kolonat

Krijohet `df_clean` nga origjinali në çdo ekzekutim. Hiqen vetëm `iucnRedListStatus` dhe `establishmentMeans`, siç përcakton strategjia e mungesave. Nënbashkësia me 29 veti e Jetës është fazë tjetër, jo kjo kopje pune. Ruhen identifikuesit, licenca dhe të gjithë rreshtat.

Flamujt `*_is_missing` përfaqësojnë mungesën pas normalizimit; `elevation_was_zero` shënon zerot origjinale pa i zëvendësuar ende. Zëvendësimi i tyre i përket hapit të mëvonshëm të vlerave të pavlefshme. Eksporti është kopje e ndërmjetme, jo checkpoint final. Çdo hap përdor `log_step()`. Për hapat me prefiksin `02_clean:` riekzekutimi përditëson të njëjtin rresht log-u; hapat e tjerë ruhen. Ky log përshkruan gjendjen e fundit të pipeline-it, jo historikun e çdo ekzekutimi.

In [9]:
section("Zbatimi: kolonat")
import hashlib
import json

raw_hash_before = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
df_clean = df_raw.copy(deep=True)
cleaning_counts = []
CLEAN_PATH = PROCESSED_DIR / "kosovo_biodiversity_clean.csv"

def record_cleaning(name, before, reason, changed=0, output="processed/kosovo_biodiversity_clean.csv"):
    log_step("02_clean:" + name, "dataset/kosovo_overall_biodiversity.csv", before,
             df_clean.shape, reason, output,
             commit="Apply cleaning decisions and preserve date precision",
             replace_existing=True)
    cleaning_counts.append({"hapi": name, "vlera_te_ndryshuara_ose_te_shenuara": int(changed),
                            "rreshta_para": before[0], "rreshta_pas": len(df_clean),
                            "kolona_para": before[1], "kolona_pas": len(df_clean.columns),
                            "arsyeja": reason})

for column in df_raw:
    new_missing = token_mask[column] & df_raw[column].notna()
    if new_missing.any():
        before = df_clean.shape
        df_clean.loc[new_missing, column] = pd.NA
        record_cleaning("tokens:" + column, before, "Token literal i mungesës -> NA; pa ndryshuar tekstet e tjera.", new_missing.sum())

before = df_clean.shape
df_clean["elevation_was_zero"] = zero_elevation.astype(bool)
record_cleaning("elevation_zero", before, "Vetëm flamur për zerot origjinale; zëvendësimi me NA shtyhet te vlerat e pavlefshme.", zero_elevation.sum())

drop_columns = [column for column, (_, decision, _) in strategies.items() if decision.startswith("delete")]
assert set(drop_columns) == {"iucnRedListStatus", "establishmentMeans"}
for column in drop_columns:
    before = df_clean.shape
    df_clean = df_clean.drop(columns=column)
    record_cleaning("drop:" + column, before, strategies[column][2], len(df_clean))

for column, (_, decision, _) in strategies.items():
    if "flag" in decision and column in df_clean:
        before = df_clean.shape
        df_clean[column + "_is_missing"] = df_clean[column].isna()
        record_cleaning("flag:" + column, before, "Flamur mungese; nuk fshihen rreshta dhe nuk imputohen vlera.", df_clean[column].isna().sum())
display(pd.DataFrame(cleaning_counts))


Zbatimi: kolonat


,hapi,vlera_te_ndryshuara_ose_te_shenuara,rreshta_para,rreshta_pas,kolona_para,kolona_pas,arsyeja
0,tokens:locality,38,46831,46831,42,42,Token literal i mungesës -> NA; pa ndryshuar tekstet e tjera.
1,tokens:recordedBy,1,46831,46831,42,42,Token literal i mungesës -> NA; pa ndryshuar tekstet e tjera.
2,tokens:identifiedBy,1,46831,46831,42,42,Token literal i mungesës -> NA; pa ndryshuar tekstet e tjera.
3,elevation_zero,3634,46831,46831,42,43,Vetëm flamur për zerot origjinale; zëvendësimi me NA shtyhet te vlerat e pavlefshme.
4,drop:iucnRedListStatus,46831,46831,46831,43,42,Redundant sipas reports/redundancy.md; jo vendim i bazuar në përqindjen e mungesës.
5,drop:establishmentMeans,46831,46831,46831,42,41,Pothuajse bosh dhe jashtë pyetjeve kryesore sipas rëndësisë; ruaj origjinalin për analizë të specializuar.
6,flag:scientificName,37,46831,46831,41,42,Flamur mungese; nuk fshihen rreshta dhe nuk imputohen vlera.
7,flag:taxonRank,37,46831,46831,42,43,Flamur mungese; nuk fshihen rreshta dhe nuk imputohen vlera.
8,flag:species,1213,46831,46831,43,44,Flamur mungese; nuk fshihen rreshta dhe nuk imputohen vlera.
9,flag:speciesKey,1213,46831,46831,44,45,Flamur mungese; nuk fshihen rreshta dhe nuk imputohen vlera.


### Dokumentimi i hapit: Kolonat

1. **Input:** dataset-i origjinal përmes `df_raw`, strategjia në seksionin paraprak dhe `reports/strategjia_e_mungesave.md`.
2. **Çka u bë:** U normalizuan tokenët e konfirmuar, u zbatuan dy fshirjet dhe flamujt e strategjisë; zerot e elevation vetëm u shënuan.
3. **Rreshta para / pas:** 46,831 / 46,831. Numrat e kolonave për çdo transformim ruhen në `processed/cleaning_log.csv`; lista para/pas në `reports/kolonat_para_pas_pastrimit.csv`.
4. **Pse u bë:** Për të zbatuar vendimet në një kopje të riprodhueshme dhe të auditueshme.
5. **Output:** `processed/cleaning_log.csv`, `reports/kolonat_para_pas_pastrimit.csv`, `reports/numerimet_e_pastrimit.csv`.
6. **Commit:** `Apply cleaning decisions and preserve date precision` (mesazh i propozuar).
7. **Decision + Rationale:** Zbatohet strategjia e mungesave; zgjedhja e vetive në notebook-un 03 mbetet e veçantë. Nuk kryhet imputim statistikor.

## Zbatimi i mungesave strukturore

**Decision + Rationale:** Sipas sqarimit të përdorueses më 23 tetor 2026, protectedArea bosh nënkupton jashtë zonave të mbrojtura të regjistruara në këtë dataset. Ky është konfirmim semantik nga përdoruesja, jo verifikim i pavarur GIS ose dokumentacion zyrtar i prodhuesit. Prandaj vetëm boshllëqet zëvendësohen me "Jashtë zonës së mbrojtur". protectedAreaDesignation bëhet "Nuk aplikohet" vetëm te të njëjtët rreshta me emër zone fillimisht bosh; një emërtim që mungon për një zonë të njohur do të mbetej unknown. Emrat ekzistues nuk ndryshohen. Flamujt is_missing rifreskohen, ndërsa protected_area_status ruan dallimin mes zonës së raportuar dhe mungesës strukturore. Kufizim: rezultati vlen sipas kuptimit dhe mbulimit të dataset-it, jo si pretendim i verifikuar për të gjithë kufijtë zyrtarë.

Për iasAppliesFrom, mungesa me iasUnionConcern=False shënohet not_applicable; data mbetet NaT.

In [10]:
section("Zbatimi: mungesat strukturore")
outside_recorded = df_clean["protectedArea"].isna()
designation_missing_before = df_clean["protectedAreaDesignation"].isna()
assert not (outside_recorded & ~designation_missing_before).any(), "Emri i zonës mungon por tipi ekziston; rishiko para zëvendësimit."
before = df_clean.shape
df_clean["protected_area_status"] = np.where(outside_recorded, "outside_recorded_protected_areas", "reported_protected_area")
record_cleaning("protected_area_status", before, "Sqarim semantik nga përdoruesja: bosh = jashtë zonave të regjistruara; jo kontroll GIS.", outside_recorded.sum())
before = df_clean.shape
df_clean.loc[outside_recorded, "protectedArea"] = "Jashtë zonës së mbrojtur"
df_clean["protectedArea_is_missing"] = df_clean["protectedArea"].isna()
record_cleaning("fill:protectedArea", before, "Zëvendësim i mungesës strukturore sipas sqarimit të përdorueses; emrat ekzistues ruhen.", outside_recorded.sum())
before = df_clean.shape
designation_fill = outside_recorded & designation_missing_before
df_clean.loc[designation_fill, "protectedAreaDesignation"] = "Nuk aplikohet"
df_clean["protectedAreaDesignation_is_missing"] = df_clean["protectedAreaDesignation"].isna()
record_cleaning("fill:protectedAreaDesignation", before, "Tipi i mbrojtjes nuk aplikohet kur pika është jashtë zonave të regjistruara.", designation_fill.sum())

concern = literal["iasUnionConcern"].str.strip().str.casefold().map({"true": True, "false": False}).astype("boolean")
missing_ias = df_clean["iasAppliesFrom"].isna()
ias_status = pd.Series("unknown", index=df_clean.index, dtype="string")
ias_status.loc[(concern.eq(False) & missing_ias).fillna(False)] = "not_applicable"
ias_status.loc[(concern.eq(True) & ~missing_ias).fillna(False)] = "reported"
ias_status.loc[(concern.eq(False) & ~missing_ias).fillna(False)] = "inconsistent"
before = df_clean.shape
df_clean["ias_date_status"] = ias_status
record_cleaning("ias_date_status", before, "Dallo not_applicable nga unknown sipas iasUnionConcern; data nuk mbushet artificialisht.", ias_status.eq("not_applicable").sum())

structural_counts = pd.DataFrame([
    {"kolona": "protectedArea", "mungesa": int(outside_recorded.sum()), "vlera_te_zevendesuara": int(outside_recorded.sum()), "not_applicable": int(outside_recorded.sum()), "unknown": int(df_clean["protectedArea"].isna().sum()), "statusi": "user_confirmed_semantics"},
    {"kolona": "protectedAreaDesignation", "mungesa": int(designation_missing_before.sum()), "vlera_te_zevendesuara": int(designation_fill.sum()), "not_applicable": int(designation_fill.sum()), "unknown": int(df_clean["protectedAreaDesignation"].isna().sum()), "statusi": "user_confirmed_semantics"},
    {"kolona": "iasAppliesFrom", "mungesa": int(missing_ias.sum()), "vlera_te_zevendesuara": 0, "not_applicable": int(ias_status.eq("not_applicable").sum()), "unknown": int(ias_status.eq("unknown").sum()), "statusi": "flag_applied"},
])
structural_counts.to_csv(REPORTS_DIR / "numerimet_e_mungesave_strukturore.csv", index=False)
structural_text = """# Vendimet për mungesat strukturore

**Decision + Rationale:** Sipas sqarimit të përdorueses më 23 tetor 2026, protectedArea bosh nënkupton jashtë zonave të mbrojtura të regjistruara në këtë dataset. Ky është konfirmim semantik nga përdoruesja, jo verifikim i pavarur GIS ose dokumentacion zyrtar i prodhuesit. Prandaj vetëm boshllëqet zëvendësohen me "Jashtë zonës së mbrojtur". protectedAreaDesignation bëhet "Nuk aplikohet" vetëm te të njëjtët rreshta me emër zone fillimisht bosh; një emërtim që mungon për një zonë të njohur do të mbetej unknown. Emrat ekzistues nuk ndryshohen. Flamujt is_missing rifreskohen, ndërsa protected_area_status ruan dallimin mes zonës së raportuar dhe mungesës strukturore. Kufizim: rezultati vlen sipas kuptimit dhe mbulimit të dataset-it, jo si pretendim i verifikuar për të gjithë kufijtë zyrtarë.

Për iasAppliesFrom, kuptimi i fushës dhe crosstab-i mbështesin not_applicable kur iasUnionConcern=False. Ky është status i zbatueshmërisë së datës, jo datë e imputuar. Rreshtat True pa datë mbeten unknown; False me datë shënohen inconsistent. Çdo datë ruhet dhe validohet në seksionin e tipeve.

""" + markdown_table(structural_counts)
(REPORTS_DIR / "vendimet_per_mungesat_strukturore.md").write_text(structural_text, encoding="utf-8")
display(Markdown(structural_text))


Zbatimi: mungesat strukturore


# Vendimet për mungesat strukturore

**Decision + Rationale:** Sipas sqarimit të përdorueses më 23 tetor 2026, protectedArea bosh nënkupton jashtë zonave të mbrojtura të regjistruara në këtë dataset. Ky është konfirmim semantik nga përdoruesja, jo verifikim i pavarur GIS ose dokumentacion zyrtar i prodhuesit. Prandaj vetëm boshllëqet zëvendësohen me "Jashtë zonës së mbrojtur". protectedAreaDesignation bëhet "Nuk aplikohet" vetëm te të njëjtët rreshta me emër zone fillimisht bosh; një emërtim që mungon për një zonë të njohur do të mbetej unknown. Emrat ekzistues nuk ndryshohen. Flamujt is_missing rifreskohen, ndërsa protected_area_status ruan dallimin mes zonës së raportuar dhe mungesës strukturore. Kufizim: rezultati vlen sipas kuptimit dhe mbulimit të dataset-it, jo si pretendim i verifikuar për të gjithë kufijtë zyrtarë.

Për iasAppliesFrom, kuptimi i fushës dhe crosstab-i mbështesin not_applicable kur iasUnionConcern=False. Ky është status i zbatueshmërisë së datës, jo datë e imputuar. Rreshtat True pa datë mbeten unknown; False me datë shënohen inconsistent. Çdo datë ruhet dhe validohet në seksionin e tipeve.

| kolona | mungesa | vlera_te_zevendesuara | not_applicable | unknown | statusi |
| --- | --- | --- | --- | --- | --- |
| protectedArea | 25762 | 25762 | 25762 | 0 | user_confirmed_semantics |
| protectedAreaDesignation | 25762 | 25762 | 25762 | 0 | user_confirmed_semantics |
| iasAppliesFrom | 46807 | 0 | 46807 | 0 | flag_applied |

### Dokumentimi i hapit: Mungesat strukturore

1. **Input:** dataset-i origjinal përmes `df_raw`, strategjia në seksionin paraprak dhe `reports/strategjia_e_mungesave.md`.
2. **Çka u bë:** U kontrolluan supozimet dhe u krijuan statuset e mbrojtjes dhe zbatueshmërisë së datës IAS.
3. **Rreshta para / pas:** 46,831 / 46,831. Numrat e kolonave për çdo transformim ruhen në `processed/cleaning_log.csv`; lista para/pas në `reports/kolonat_para_pas_pastrimit.csv`.
4. **Pse u bë:** Për të zbatuar vendimet në një kopje të riprodhueshme dhe të auditueshme.
5. **Output:** `reports/numerimet_e_mungesave_strukturore.csv`, `reports/vendimet_per_mungesat_strukturore.md`, cleaning log.
6. **Commit:** `Apply cleaning decisions and preserve date precision` (mesazh i propozuar).
7. **Decision + Rationale:** protectedArea plotësohet sipas sqarimit semantik të përdorueses; kjo nuk është verifikim GIS. not_applicable është flamur, jo zëvendësim i datës IAS.

## Konvertimi i datave

`eventDate` ruhet si tekst burimor. `event_start`/`event_end` përfaqësojnë kufij kalendarikë: për vitin, 1 janar–31 dhjetor; për muajin, dita e parë–e fundit; për një ditë, të dy kufijtë janë ajo ditë. Këta kufij nuk imputojnë ditë të vëzhguara. Për intervalin `a/b` ruhet fillimi i a dhe fundi i b, si dhe precizioni i secilit skaj. `date_precision` ka day/month/year/interval; bosh kur mungon ose është i pavlefshëm.

Ora dhe timezone ruhen te teksti origjinal; kolonat e reja punojnë në nivel dite sipas datës kalendarike të raportuar, pa konvertim timezone dhe pa ndryshim dite. Data të pavlefshme, formate të panjohura dhe intervale të përmbysura shënohen, nuk pranohen në heshtje. year/month/day nuk mbushen nga kufijtë e intervaleve ose datat e pjesshme.

In [11]:
section("Konvertimi i datave")
from datetime import date, datetime
import calendar

def parse_endpoint(value):
    if re.fullmatch(r"\d{4}", value):
        year = int(value)
        return date(year, 1, 1), date(year, 12, 31), "year"
    if re.fullmatch(r"\d{4}-\d{2}", value):
        year, month = map(int, value.split("-"))
        return date(year, month, 1), date(year, month, calendar.monthrange(year, month)[1]), "month"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        day = date.fromisoformat(value)
        return day, day, "day"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}(?::\d{2}(?:\.\d+)?)?(?:Z|[+-]\d{2}:\d{2})?", value):
        day = datetime.fromisoformat(value.replace("Z", "+00:00")).date()
        return day, day, "day"
    raise ValueError("unsupported_format")

def parse_event(value):
    if pd.isna(value):
        return (None, None, None, None, None, "missing")
    try:
        parts = str(value).strip().split("/")
        if len(parts) not in (1, 2):
            raise ValueError("invalid_interval")
        start, end, start_precision = parse_endpoint(parts[0])
        end_precision = start_precision
        precision = start_precision
        if len(parts) == 2:
            _, end, end_precision = parse_endpoint(parts[1])
            precision = "interval"
        if start > end:
            raise ValueError("reversed_interval")
        return (start, end, precision, start_precision, end_precision, "parsed")
    except (ValueError, OverflowError) as error:
        return (None, None, None, None, None, "invalid: " + str(error))

before = df_clean.shape
parsed_dates = pd.DataFrame(df_clean["eventDate"].map(parse_event).tolist(), index=df_clean.index,
                            columns=["event_start", "event_end", "date_precision", "event_start_precision", "event_end_precision", "date_parse_status"])
for column in ["event_start", "event_end"]:
    parsed_dates[column] = pd.to_datetime(parsed_dates[column], errors="raise")
for column in parsed_dates:
    df_clean[column] = parsed_dates[column]
record_cleaning("event_dates", before, "Ruaj dy skajet dhe precision; kufijtë kalendarikë nuk janë data të imputuara.", parsed_dates["date_parse_status"].eq("parsed").sum())
format_labels = df_clean["eventDate"].astype("string").str.replace(r"\d", "9", regex=True).fillna("[missing]")
date_formats = format_labels.value_counts().rename_axis("Formati").reset_index(name="n")
date_formats.to_csv(REPORTS_DIR / "formatet_e_datave.csv", index=False)
invalid_dates = df_clean["date_parse_status"].str.startswith("invalid")
df_clean.loc[invalid_dates, ["gbifID", "eventDate", "date_parse_status"]].to_csv(REPORTS_DIR / "problemet_e_parsimit_te_datave.csv", index=False)
display(date_formats)
display(df_clean["date_precision"].value_counts(dropna=False))


Konvertimi i datave


,Formati,n
0,9999-99-99,37945
1,9999-99-99T99:99:99,2554
2,9999-99-99T99:99,2417
3,[missing],2190
4,9999-99-99T99:99:99.999Z,595
5,9999,409
6,9999-99-99/9999-99-99,398
7,9999-99,173
8,9999-99-99T99:99:99Z,67
9,9999/9999,46


date_precision
day         43612
NaN          2190
interval      447
year          409
month         173
Name: count, dtype: int64

### Dokumentimi i hapit: Datat

1. **Input:** dataset-i origjinal përmes `df_raw`, strategjia në seksionin paraprak dhe `reports/strategjia_e_mungesave.md`.
2. **Çka u bë:** U parsuan formatet ISO dhe intervalet; u ruajt teksti burimor dhe precizioni i skajeve.
3. **Rreshta para / pas:** 46,831 / 46,831. Numrat e kolonave për çdo transformim ruhen në `processed/cleaning_log.csv`; lista para/pas në `reports/kolonat_para_pas_pastrimit.csv`.
4. **Pse u bë:** Për të zbatuar vendimet në një kopje të riprodhueshme dhe të auditueshme.
5. **Output:** `reports/formatet_e_datave.csv`, `reports/problemet_e_parsimit_te_datave.csv`, event_start/event_end/date_precision dhe precizionet e skajeve.
6. **Commit:** `Apply cleaning decisions and preserve date precision` (mesazh i propozuar).
7. **Decision + Rationale:** Datat e pjesshme përfaqësohen me kufij, pa shpikur komponentë year/month/day. Intervalet nuk reduktohen vetëm në fillim.

## Konvertimi i tipeve të tjera dhe eksporti

Përdoren tipet e `reports/data_types.md` për numrat dhe kategoritë nominale, me përjashtimet e dokumentuara: IUCN është category pa renditje dhe ruan NE/DD; taxonRank ruan UNRANKED si kategori pa renditje. Kodimi ordinal mbetet te Jeta. `eventDate` mbetet tekst sepse të dy skajet dhe precision ruhen në kolonat e reja.

CSV nuk ruan Int64/category/datetime si metadata. `processed/kosovo_biodiversity_clean_schema.json` dhe `load_clean_dataset()` më poshtë mundësojnë rikthimin e tipeve. Skedari i pastruar rindërtohet gjithmonë nga origjinali. Përmbajtja e raw CSV kontrollohet me SHA-256 para/pas.

In [12]:
section("Konvertimi i tipeve dhe eksporti")
dtypes_before = df_clean.dtypes.astype(str).copy()
integer_columns = ["year", "month", "day", "taxonKey", "speciesKey", "individualCount"]
for column in integer_columns:
    before = df_clean.shape
    numeric = pd.to_numeric(df_clean[column], errors="raise")
    assert numeric.dropna().mod(1).eq(0).all(), f"Vlerë jo e plotë: {column}"
    df_clean[column] = numeric.astype("Int64")
    record_cleaning("dtype:" + column, before, "Numër i plotë nullable; mungesat ruhen.", df_clean[column].notna().sum())

for column in ["strictlyProtected", "iasUnionConcern"]:
    before = df_clean.shape
    values = df_clean[column].astype("string").str.strip().str.casefold()
    assert values.dropna().isin(["true", "false"]).all(), f"Boolean i panjohur: {column}"
    converted = values.map({"true": True, "false": False}).astype("boolean")
    df_clean[column] = converted.astype(bool) if converted.notna().all() else converted
    record_cleaning("dtype:" + column, before, "TRUE/FALSE -> bool; nullable boolean nëse ka mungesa.", converted.notna().sum())

before = df_clean.shape
df_clean["iasAppliesFrom"] = pd.to_datetime(df_clean["iasAppliesFrom"], format="%Y-%m-%d", errors="raise")
record_cleaning("dtype:iasAppliesFrom", before, "Datë ISO; mungesa strukturore ruhet si NaT, statusi veçmas.", df_clean["iasAppliesFrom"].notna().sum())
categorical_columns = "scientificName species vernacularName kingdom phylum class order family genus taxonRank iucnRedListCategory municipality district protectedArea protectedAreaDesignation basisOfRecord institutionCode collectionCode datasetKey license protected_area_status ias_date_status date_precision event_start_precision event_end_precision date_parse_status".split()
for column in categorical_columns:
    before = df_clean.shape
    df_clean[column] = df_clean[column].astype(pd.CategoricalDtype(ordered=False))
    record_cleaning("dtype:" + column, before, "Kategori pa renditje; asnjë vlerë nuk hiqet dhe nuk bëhet kodim ordinal.", df_clean[column].notna().sum())




Konvertimi i tipeve dhe eksporti


## Plotësimi i year/month nga eventDate

Plotësohen vetëm qelizat bosh nga data të plota, të parsueshme. Datat vetëm me vit ose vit-muaj përjashtohen nga ky hap, sipas kërkesës për të mos imputuar nga data të pjesshme. Për intervalet kërkohen dy skaje me precision day dhe brenda të njëjtit muaj e vit; intervalet ndërmujore ose ndërvjetore përjashtohen për të dy komponentët.

Nëse një komponent ekzistues year/month bie ndesh me datën, rreshti ruhet pa plotësim dhe raportohet. Vlerat ekzistuese dhe day nuk ndryshohen. Flamujt year_was_filled/month_was_filled identifikojnë rikuperimin determinist; year_is_missing/month_is_missing rifreskohen pas plotësimit.

In [13]:
section("Plotësimi i year/month")

def year_month_candidates(frame):
    start, end = frame["event_start"], frame["event_end"]
    valid = frame["date_parse_status"].eq("parsed") & start.notna() & end.notna()
    full_day = frame["date_precision"].eq("day")
    same_month_interval = (
        frame["date_precision"].eq("interval")
        & frame["event_start_precision"].eq("day")
        & frame["event_end_precision"].eq("day")
        & start.dt.year.eq(end.dt.year) & start.dt.month.eq(end.dt.month)
    )
    eligible = valid & (full_day | same_month_interval)
    derived = pd.DataFrame({"year": start.dt.year.astype("Int64"), "month": start.dt.month.astype("Int64")})
    conflict = pd.Series(False, index=frame.index)
    for column in ["year", "month"]:
        conflict |= (frame[column].notna() & frame[column].ne(derived[column])).fillna(False)
    reason = pd.Series("missing_or_invalid_date", index=frame.index, dtype="string")
    reason.loc[valid & ~eligible] = "partial_or_spanning_interval"
    reason.loc[eligible & conflict] = "existing_component_conflict"
    reason.loc[eligible & ~conflict] = "unambiguous"
    return derived, reason

year_month_before = df_clean[["year", "month"]].copy()
derived_components, recovery_reason = year_month_candidates(df_clean)
recovery_rows, recovery_details = [], []
for column in ["year", "month"]:
    before = df_clean.shape
    was_missing = year_month_before[column].isna()
    fill_mask = was_missing & recovery_reason.eq("unambiguous")
    df_clean.loc[fill_mask, column] = derived_components.loc[fill_mask, column]
    df_clean[column] = df_clean[column].astype("Int64")
    df_clean[column + "_was_filled"] = fill_mask.astype(bool)
    df_clean[column + "_is_missing"] = df_clean[column].isna()
    record_cleaning("recover:" + column, before,
                    "Plotësim determinist nga eventDate e plotë ose interval brenda të njëjtit muaj; pa mbishkrim.",
                    fill_mask.sum())
    recovery_rows.append({"kolona": column, "mungesa_para": int(was_missing.sum()),
                          "te_plotesuara": int(fill_mask.sum()), "mungesa_pas": int(df_clean[column].isna().sum()),
                          "perqindja_para": round(was_missing.mean() * 100, 2),
                          "perqindja_pas": round(df_clean[column].isna().mean() * 100, 2)})
    details = df_clean.loc[was_missing, ["gbifID", "eventDate", "date_precision", "event_start", "event_end"]].copy()
    details["kolona"] = column
    details["vlera_para"] = pd.NA
    details["vlera_pas"] = df_clean.loc[was_missing, column]
    details["u_plotesua"] = fill_mask.loc[was_missing]
    details["arsyeja"] = recovery_reason.loc[was_missing]
    recovery_details.append(details)
    pd.testing.assert_series_equal(df_clean.loc[~was_missing, column], year_month_before.loc[~was_missing, column])
    assert was_missing.sum() - df_clean[column].isna().sum() == fill_mask.sum()

recovery_summary = pd.DataFrame(recovery_rows)
recovery_audit = pd.concat(recovery_details, ignore_index=True)
recovery_summary.to_csv(REPORTS_DIR / "plotesimi_year_month_para_pas.csv", index=False)
recovery_audit.to_csv(REPORTS_DIR / "gjurma_e_plotesimit_year_month.csv", index=False)
recovery_report = """# Plotësimi i year/month

**Decision + Rationale:** Plotësimi është rikuperim determinist, jo imputim statistikor. Përdoren vetëm data të plota ose intervale me dy skaje të plota brenda të njëjtit muaj dhe vit. Data vetëm me vit ose vit-muaj përjashtohet sipas kërkesës së detyrës. Intervalet që kalojnë muaj/vite nuk plotësojnë as year, as month. Kufijtë artificialë të datave të pjesshme nuk përdoren si evidencë. Një konflikt me year/month ekzistues e ndalon plotësimin e rreshtit; vlerat ekzistuese nuk korrigjohen automatikisht. day mbetet i pandryshuar.

""" + markdown_table(recovery_summary) + """

`gjurma_e_plotesimit_year_month.csv` përmban çdo qelizë year/month që ishte bosh, ID-në, eventDate, kufijtë, rezultatin dhe arsyen e plotësimit ose përjashtimit. `year_was_filled`/`month_was_filled` tregojnë vlerat e rikuperuara. Flamujt `*_is_missing` tregojnë mungesën aktuale pas plotësimit.

## Dokumentimi i detyrës

1. **Input:** df_clean pas parsimit të eventDate dhe konvertimit të tipeve; burimi dataset/kosovo_overall_biodiversity.csv.
2. **Çka u bë:** U plotësuan vetëm year/month të derivueshme pa paqartësi dhe u regjistruan hapat me log_step().
3. **Rreshta para/pas:** 46,831 / 46,831; dy flamuj shtesë. Numrat e kolonave janë në processed/cleaning_log.csv.
4. **Pse u bë:** Për të rikuperuar informacionin kohor të raportuar pa shpikur data dhe pa mbishkruar vlera.
5. **Output:** plotesimi_year_month_para_pas.csv, gjurma_e_plotesimit_year_month.csv, flamujt dhe dataset-i i ndërmjetëm.
6. **Commit:** Fill unambiguous missing year and month values (mesazh i propozuar).

Duplikatet nuk trajtohen në këtë hap; varen nga analiza e Dafinës.
"""
(REPORTS_DIR / "vendimi_per_plotesimin_year_month.md").write_text(recovery_report, encoding="utf-8")
display(Markdown(recovery_report))


Plotësimi i year/month


# Plotësimi i year/month

**Decision + Rationale:** Plotësimi është rikuperim determinist, jo imputim statistikor. Përdoren vetëm data të plota ose intervale me dy skaje të plota brenda të njëjtit muaj dhe vit. Data vetëm me vit ose vit-muaj përjashtohet sipas kërkesës së detyrës. Intervalet që kalojnë muaj/vite nuk plotësojnë as year, as month. Kufijtë artificialë të datave të pjesshme nuk përdoren si evidencë. Një konflikt me year/month ekzistues e ndalon plotësimin e rreshtit; vlerat ekzistuese nuk korrigjohen automatikisht. day mbetet i pandryshuar.

| kolona | mungesa_para | te_plotesuara | mungesa_pas | perqindja_para | perqindja_pas |
| --- | --- | --- | --- | --- | --- |
| year | 2268 | 0 | 2268 | 4.84 | 4.84 |
| month | 2867 | 0 | 2867 | 6.12 | 6.12 |

`gjurma_e_plotesimit_year_month.csv` përmban çdo qelizë year/month që ishte bosh, ID-në, eventDate, kufijtë, rezultatin dhe arsyen e plotësimit ose përjashtimit. `year_was_filled`/`month_was_filled` tregojnë vlerat e rikuperuara. Flamujt `*_is_missing` tregojnë mungesën aktuale pas plotësimit.

## Dokumentimi i detyrës

1. **Input:** df_clean pas parsimit të eventDate dhe konvertimit të tipeve; burimi dataset/kosovo_overall_biodiversity.csv.
2. **Çka u bë:** U plotësuan vetëm year/month të derivueshme pa paqartësi dhe u regjistruan hapat me log_step().
3. **Rreshta para/pas:** 46,831 / 46,831; dy flamuj shtesë. Numrat e kolonave janë në processed/cleaning_log.csv.
4. **Pse u bë:** Për të rikuperuar informacionin kohor të raportuar pa shpikur data dhe pa mbishkruar vlera.
5. **Output:** plotesimi_year_month_para_pas.csv, gjurma_e_plotesimit_year_month.csv, flamujt dhe dataset-i i ndërmjetëm.
6. **Commit:** Fill unambiguous missing year and month values (mesazh i propozuar).

Duplikatet nuk trajtohen në këtë hap; varen nga analiza e Dafinës.


### Dokumentimi i hapit: Plotësimi i year/month

1. **Input:** eventDate, event_start/event_end, precision dhe year/month nga df_clean.
2. **Çka u bë:** Plotësim i qelizave bosh vetëm nga data të plota pa konflikt; regjistrim i arsyeve dhe flamujve.
3. **Rreshta para/pas:** 46,831 / 46,831; kolona 77 / 79. Çdo hap gjendet në cleaning_log.csv.
4. **Pse u bë:** Rikuperim i informacionit kohor të raportuar.
5. **Output:** reports/plotesimi_year_month_para_pas.csv, reports/gjurma_e_plotesimit_year_month.csv, reports/vendimi_per_plotesimin_year_month.md.
6. **Commit:** Fill unambiguous missing year and month values (mesazh i propozuar).
7. **Decision + Rationale:** Pa data të pjesshme, intervale ndërmujore/ndërvjetore ose konflikte; pa mbishkrim të year/month ekzistues dhe pa ndryshim të day.

## Eksporti i kopjes së ndërmjetme

Eksporti përfshin edhe year/month e rikuperuara dhe flamujt e tyre.

In [14]:
dtypes_report_clean = pd.DataFrame({"kolona": df_clean.columns, "dtype_para": [dtypes_before.get(c, "[new]") for c in df_clean],
                                  "dtype_pas": df_clean.dtypes.astype(str).values,
                                  "dtype_origjinal": [str(df_raw[c].dtype) if c in df_raw else "[new]" for c in df_clean]})
dtypes_report_clean.to_csv(REPORTS_DIR / "tipet_para_pas_pastrimit.csv", index=False)
column_inventory = pd.DataFrame({"kolona": list(df_raw.columns) + [c for c in df_clean if c not in df_raw]})
column_inventory["para"] = column_inventory["kolona"].isin(df_raw.columns)
column_inventory["pas"] = column_inventory["kolona"].isin(df_clean.columns)
column_inventory["veprimi"] = np.select([~column_inventory["pas"], ~column_inventory["para"]], ["removed", "added"], default="kept")
column_inventory.to_csv(REPORTS_DIR / "kolonat_para_pas_pastrimit.csv", index=False)

assert df_clean["gbifID"].equals(df_raw["gbifID"])
assert not df_clean["iucnRedListCategory"].cat.ordered
assert df_clean["iucnRedListCategory"].astype("string").equals(df_raw["iucnRedListCategory"].astype("string"))
pd.testing.assert_frame_equal(df_raw, pd.read_csv(DATA_PATH, low_memory=False))
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == raw_hash_before

df_clean.to_csv(CLEAN_PATH, index=False)
schema = {"source": "dataset/kosovo_overall_biodiversity.csv", "source_sha256": raw_hash_before,
          "rows": len(df_clean), "columns": {}}
for column in df_clean:
    dtype = df_clean[column].dtype
    spec = {"dtype": str(dtype)}
    if isinstance(dtype, pd.CategoricalDtype):
        spec.update(categories=df_clean[column].cat.categories.tolist(), ordered=dtype.ordered, categories_dtype=str(df_clean[column].cat.categories.dtype))
    schema["columns"][column] = spec
SCHEMA_PATH = PROCESSED_DIR / "kosovo_biodiversity_clean_schema.json"
SCHEMA_PATH.write_text(json.dumps(schema, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def load_clean_dataset(path=CLEAN_PATH, schema_path=SCHEMA_PATH):
    saved_schema = json.loads(Path(schema_path).read_text(encoding="utf-8"))
    data = pd.read_csv(path, dtype="string", keep_default_na=False, na_values=[""])
    assert list(data.columns) == list(saved_schema["columns"])
    for column, spec in saved_schema["columns"].items():
        dtype = spec["dtype"]
        if dtype == "category":
            assert data[column].dropna().isin(spec["categories"]).all()
            categories = pd.Index(spec["categories"], dtype=spec["categories_dtype"])
            data[column] = data[column].astype(pd.CategoricalDtype(categories, ordered=spec["ordered"]))
        elif dtype.startswith("datetime64"):
            data[column] = pd.to_datetime(data[column], errors="raise").astype(dtype)
        elif dtype in ("bool", "boolean"):
            assert data[column].dropna().isin(["True", "False"]).all()
            data[column] = data[column].map({"True": True, "False": False}).astype(dtype)
        else:
            data[column] = data[column].astype(dtype)
    assert len(data) == saved_schema["rows"]
    return data

reloaded_clean = load_clean_dataset()
pd.testing.assert_frame_equal(df_clean, reloaded_clean, check_exact=False, rtol=1e-12, atol=1e-12)
record_cleaning("export", df_clean.shape, "CSV e veçantë + schema; verifikuar rikthimi i tipeve dhe hash-i i origjinalit.", 0)
pd.DataFrame(cleaning_counts).to_csv(REPORTS_DIR / "numerimet_e_pastrimit.csv", index=False)
display(dtypes_report_clean)
print(f"Dataset-i origjinal: {df_raw.shape}; dataset-i i pastruar: {df_clean.shape}")
print(f"Ruajtur: {CLEAN_PATH}")
print("protectedArea: zëvendësimi u zbatua sipas sqarimit të përdorueses; shih vendimet_per_mungesat_strukturore.md.")

,kolona,dtype_para,dtype_pas,dtype_origjinal
0,gbifID,int64,int64,int64
1,scientificName,str,category,str
2,species,str,category,str
3,vernacularName,str,category,str
4,kingdom,str,category,str
...,...,...,...,...
74,event_start_precision,str,category,[new]
75,event_end_precision,str,category,[new]
76,date_parse_status,str,category,[new]
77,year_was_filled,[new],bool,[new]


Dataset-i origjinal: (46831, 42); dataset-i i pastruar: (46831, 79)
Ruajtur: /Users/delizakadriu/Desktop/FIEK/Master/Pergatitja_e_te_dhenave/PVDH_Kosovo-Biodiversity-Analysis/processed/kosovo_biodiversity_clean.csv
protectedArea: zëvendësimi u zbatua sipas sqarimit të përdorueses; shih vendimet_per_mungesat_strukturore.md.


### Dokumentimi i hapit: Tipet dhe eksporti

1. **Input:** dataset-i origjinal përmes `df_raw`, strategjia në seksionin paraprak dhe `reports/strategjia_e_mungesave.md`.
2. **Çka u bë:** U konvertuan numrat, boolean-et, datat dhe kategoritë; u eksportua kopja me schema dhe u kontrollua rileximi.
3. **Rreshta para / pas:** 46,831 / 46,831. Numrat e kolonave për çdo transformim ruhen në `processed/cleaning_log.csv`; lista para/pas në `reports/kolonat_para_pas_pastrimit.csv`.
4. **Pse u bë:** Për të zbatuar vendimet në një kopje të riprodhueshme dhe të auditueshme.
5. **Output:** `processed/kosovo_biodiversity_clean.csv`, `processed/kosovo_biodiversity_clean_schema.json`, `reports/tipet_para_pas_pastrimit.csv` dhe log-u.
6. **Commit:** `Apply cleaning decisions and preserve date precision` (mesazh i propozuar).
7. **Decision + Rationale:** IUCN mbetet category pa renditje; NE/DD ruhen. Origjinali nuk ndryshohet. Rileximi me schema ruan tipet që CSV vetëm nuk i ruan.

## Duplikatet: vendimi dhe rreshtat e përzgjedhur

Varësia është analiza e Dafinës: `processed/duplicate_candidates.csv`, `reports/uniqueness.md` dhe `reports/01_duplikatet_shqyrtimi.csv`, commit `d2708c640de830014edf4c4f35a528849c98455f`. Për riprodhueshmëri lexohen nga ky version i historikut Git, pa ndryshuar notebook-un ose skedarët e saj. Nëse ky commit mungon lokalisht, ekzekuto `git fetch origin` para notebook-ut. Importimi nuk bën merge/push.

Kriteri fillestar është species + eventDate origjinale + koordinata të sakta + recordedBy i normalizuar. Vendimi ruan përjashtimet e Dafinës dhe shton kontrolle për konflikte: nuk mjafton vetëm barazia e numrit të individëve ose ndryshimi i datasetKey. Për grupet ndërmjet burimeve kërkohet gjithashtu konfirmim pozitiv në shqyrtimin e Dafinës dhe precizion ditor. Nuk fshihet asnjë rresht në këtë seksion.

In [15]:
section("Duplikatet: vendimi")
import io
import subprocess

DUPLICATE_SOURCE_COMMIT = "d2708c640de830014edf4c4f35a528849c98455f"

def read_duplicate_source(relative_path):
    try:
        return subprocess.check_output(["git", "show", f"{DUPLICATE_SOURCE_COMMIT}:{relative_path}"], cwd=ROOT)
    except subprocess.CalledProcessError as error:
        raise RuntimeError("Mungon varësia Git; ekzekuto git fetch origin dhe provo përsëri.") from error

candidate_bytes = read_duplicate_source("processed/duplicate_candidates.csv")
review_bytes = read_duplicate_source("reports/01_duplikatet_shqyrtimi.csv")
source_report = read_duplicate_source("reports/uniqueness.md").decode("utf-8")
assert hashlib.sha256(read_duplicate_source("dataset/kosovo_overall_biodiversity.csv")).hexdigest() == raw_hash_before
candidates = pd.read_csv(io.BytesIO(candidate_bytes), low_memory=False)
manual_review = pd.read_csv(io.BytesIO(review_bytes))
assert candidates["gbifID"].is_unique and df_raw["gbifID"].is_unique
assert set(candidates["gbifID"]) <= set(df_raw["gbifID"])

raw_by_id = df_raw.set_index("gbifID")
clean_by_id = df_clean.set_index("gbifID")
literal_by_id = literal.set_index(df_raw["gbifID"])
effective_by_id = effective_missing.set_index(df_raw["gbifID"])
key_columns = ["species", "eventDate", "decimalLatitude", "decimalLongitude"]
for column in key_columns:
    pd.testing.assert_series_equal(candidates.set_index("gbifID")[column],
                                   raw_by_id.loc[candidates["gbifID"], column], check_names=False)

# Rezultati i plotësisë nuk përfshin ID, metadata botimi, kolona redundante ose flamuj të derivuar.
score_columns = "scientificName species kingdom phylum class order family genus taxonRank eventDate year month day decimalLatitude decimalLongitude coordinateUncertaintyInMeters elevation locality municipality district protectedArea protectedAreaDesignation strictlyProtected basisOfRecord individualCount recordedBy identifiedBy iucnRedListCategory".split()
completeness = (~effective_by_id[score_columns]).sum(axis=1)
positive_review_ids = set(manual_review.loc[manual_review["vendimi_manual"].eq("duplikat"), "gbifID"])
specimen_types = {"PRESERVED_SPECIMEN", "MATERIAL_SAMPLE", "FOSSIL_SPECIMEN", "LIVING_SPECIMEN"}
group_decisions, ranked_rows, deletion_rows = [], [], []

for group_id, group in candidates.groupby("dup_group", sort=True):
    ids = group["gbifID"].tolist()
    records = raw_by_id.loc[ids]
    clean_records = clean_by_id.loc[ids]
    assert len(group) == group["group_size"].iloc[0]
    assert group["group_size"].nunique() == 1 and group["duplikat_i_propozuar"].nunique() == 1
    assert records[key_columns].notna().all().all()
    assert all(records[column].nunique() == 1 for column in key_columns)
    observers = literal_by_id.loc[ids, "recordedBy"].str.strip().str.replace(r"\s+", " ", regex=True).str.casefold()
    assert observers.nunique() == 1
    exact_day = clean_records["date_precision"].eq("day").all()
    observer_unknown = effective_by_id.loc[ids, "recordedBy"].any()
    count_conflict = records["individualCount"].nunique(dropna=False) > 1
    taxon_conflict = records["scientificName"].nunique(dropna=False) > 1
    basis_conflict = records["basisOfRecord"].nunique(dropna=False) > 1
    coarse_grid = records["coordinateUncertaintyInMeters"].eq(7071).any()
    multi_source = records["datasetKey"].nunique() > 1
    reviewed_positive = bool(set(ids) & positive_review_ids)
    distinct_specimens = records["basisOfRecord"].isin(specimen_types).any() and records["catalogNumber"].dropna().nunique() > 1
    proposed = bool(group["duplikat_i_propozuar"].iloc[0])
    blockers = []
    if not proposed:
        blockers.append("jo_duplikat_sipas_Dafines")
    if observer_unknown:
        blockers.append("vezhgues_i_panjohur")
    if not exact_day:
        blockers.append("date_jo_ditore")
    if coarse_grid:
        blockers.append("qender_rrjete_7071m")
    if count_conflict:
        blockers.append("individualCount_konfliktual")
    if taxon_conflict or basis_conflict:
        blockers.append("takson_ose_basisOfRecord_konfliktual")
    if multi_source and not reviewed_positive:
        blockers.append("burime_te_ndryshme_pa_konfirmim")
    if distinct_specimens and not (multi_source and reviewed_positive):
        blockers.append("ekzemplare_me_kode_te_ndryshme")
    approved = not blockers
    ranked = pd.DataFrame({"gbifID": ids, "fusha_te_plota": completeness.loc[ids].to_numpy()})
    ranked = ranked.sort_values(["fusha_te_plota", "gbifID"], ascending=[False, True], kind="stable")
    keeper = int(ranked.iloc[0]["gbifID"]) if approved else None
    reason = "pranohet_sipas_rregullit_operacional" if approved else "; ".join(blockers)
    group_decisions.append({"grupi": group_id, "rreshta": len(group), "propozimi_Dafina": proposed,
                            "pranohet": approved, "gbifID_per_ruajtje": keeper,
                            "rreshta_per_fshirje": len(group) - 1 if approved else 0,
                            "lloji_Dafina": group["lloji"].iloc[0], "arsyeja": reason})
    for rank, row in enumerate(ranked.itertuples(index=False), start=1):
        remove = approved and row.gbifID != keeper
        item = {"grupi": group_id, "gbifID": int(row.gbifID), "renditja": rank,
                "fusha_te_plota": row.fusha_te_plota, "gbifID_per_ruajtje": keeper,
                "veprimi": "per_fshirje" if remove else "ruaj", "arsyeja": reason}
        ranked_rows.append(item)
        if remove:
            deletion_rows.append(item)

duplicate_decisions = pd.DataFrame(group_decisions)
ranked_candidates = pd.DataFrame(ranked_rows)
to_remove = pd.DataFrame(deletion_rows, columns=ranked_candidates.columns)
for table in [duplicate_decisions, ranked_candidates, to_remove]:
    table["gbifID_per_ruajtje"] = table["gbifID_per_ruajtje"].astype("Int64")
duplicate_decisions.to_csv(REPORTS_DIR / "vendimet_e_grupeve_duplikate.csv", index=False)
ranked_candidates.to_csv(REPORTS_DIR / "renditja_e_kandidateve_duplikate.csv", index=False)
to_remove.to_csv(PROCESSED_DIR / "rreshtat_per_fshirje_duplikate.csv", index=False)
assert to_remove["gbifID"].is_unique
assert not set(to_remove["gbifID"]) & set(to_remove["gbifID_per_ruajtje"])
assert len(to_remove) == duplicate_decisions["rreshta_per_fshirje"].sum()
assert len(df_clean) == len(df_raw) and df_clean["gbifID"].equals(df_raw["gbifID"])

decision_text = f"""# Vendimi për duplikatet

## Evidenca

Analiza e Dafinës: commit `{DUPLICATE_SOURCE_COMMIT}`, `reports/uniqueness.md`, `reports/01_duplikatet_shqyrtimi.csv` dhe `processed/duplicate_candidates.csv`. Këto lexohen me git show nga versioni i fiksuar, pa i ndryshuar. Nëse mungon commit-i, kërkohet git fetch origin. SHA-256 e kandidatëve: `{hashlib.sha256(candidate_bytes).hexdigest()}`. Dataset-i burimor i atij commit-i përputhet me origjinalin lokal.

Ka {len(candidates):,} rreshta në {candidates['dup_group'].nunique():,} grupe kandidatë. Dafina propozon {(candidates[candidates['duplikat_i_propozuar']].groupby('dup_group').size() - 1).sum():,} rreshta të tepërt. Vendimi këtu pranon {int(duplicate_decisions['pranohet'].sum()):,} grupe dhe liston {len(to_remove):,} rreshta për fshirjen e ardhshme. Asnjë rresht nuk është fshirë.

## Decision + Rationale

Njësia e analizës është vëzhgimi/grumbullimi, jo numri i publikimeve të tij. Kriteri kandidat është species + eventDate origjinale + koordinata pa rrumbullakim + recordedBy me hapësira të normalizuara dhe casefold. Kandidatët me çelës jo të plotë nuk bashkohen; identitetet alternative të të njëjtit person nuk hamendësohen. Nuk bashkohen timestamp-e të ndryshme vetëm pse bien në të njëjtën ditë.

Pranohen vetëm grupet e propozuara nga Dafina që kanë precizion ditor, vëzhgues të njohur, scientificName dhe basisOfRecord konsistent, pa qendër rrjete 7071 m dhe pa konflikt në individualCount. Intervalet/datat e pjesshme ruhen. Specimenët me catalogNumber të ndryshëm ruhen, përveç grupeve ndërburimore të konfirmuara shprehimisht nga shqyrtimi i Dafinës. Vetëm fakti që datasetKey ndryshon nuk provon dyfishim. Përjashtimet ndaj propozimit të saj dokumentohen grup më grup në vendimet_e_grupeve_duplikate.csv.

Ky është rregull operacional për deduplikim të regjistrimeve të vëzhgimit; barazia e vendit, datës, vëzhguesit dhe numrit nuk provon identitetin biologjik të individëve. Vizitat e përsëritura mund të jenë vëzhgime të vlefshme. Kjo pasiguri ruhet në këtë arsyetim dhe lista është e auditueshme para largimit; nuk pretendohet verifikim manual i çdo grupi.

## Cili rresht ruhet

Ruhet rreshti me numrin më të madh të fushave semantikisht të plotësuara në këtë listë fikse: {', '.join(score_columns)}. Mungesat efektive përfshijnë tokenët e konfirmuar dhe elevation=0 për vlerësim; kjo nuk e zëvendëson ende elevation në dataset. ID-të, metadata e botimit, flamujt e rinj dhe iucnRedListStatus redundant nuk japin pikë. Në barazim ruhet gbifID numerik më i vogël, vetëm për qëndrueshmëri, jo sepse është më cilësor. Mungesat e rreshtit të mbajtur nuk mbushen nga rreshta të tjerë në këtë detyrë.

## individualCount

Kur numërimet ndryshojnë, ruhet i gjithë grupi. Një numër i dhënë dhe një mungesë trajtohen gjithashtu si konflikt. Kur numërimet janë të njëjta ruhet numri i rreshtit të përzgjedhur, pa mbledhje, mesatare apo maksimum. Kur të gjitha mungojnë, mbeten NaN. Numërimet e ndryshme mund të jenë vizita të ndryshme: as ruajtja e tyre nuk autorizon mbledhjen e tyre si bollëk biologjik.

## Dokumentimi i detyrës

1. **Input:** kandidatët dhe shqyrtimi i Dafinës nga commit-i i mësipërm, df_raw dhe df_clean.
2. **Çka u bë:** U vendos rregulli, përjashtimet dhe renditja e kandidatëve; u përgatit lista e rreshtave për fshirje.
3. **Rreshta para/pas:** {len(df_clean):,} / {len(df_clean):,}; **kolona para/pas:** {df_clean.shape[1]} / {df_clean.shape[1]}.
4. **Pse u bë:** Për të shmangur fshirjen e ekzemplarëve/numërimeve të ndryshme dhe për të pasur përzgjedhje të auditueshme.
5. **Output:** reports/vendimi_per_duplikatet.md, reports/vendimet_e_grupeve_duplikate.csv, reports/renditja_e_kandidateve_duplikate.csv, processed/rreshtat_per_fshirje_duplikate.csv.
6. **Commit:** Document duplicate decisions and select removal candidates (mesazh i propozuar).

Largimi i rreshtave është detyrë tjetër. cleaning_log.csv nuk merr transformim fshirjeje për këtë vendim, sepse df_clean nuk ndryshon. protectedArea është plotësuar sipas sqarimit semantik të përdorueses; shih vendimet_per_mungesat_strukturore.md për burimin dhe kufizimet.
"""
(REPORTS_DIR / "vendimi_per_duplikatet.md").write_text(decision_text, encoding="utf-8")
display(Markdown(decision_text))
display(duplicate_decisions.groupby(["propozimi_Dafina", "pranohet"])["rreshta"].agg(["size", "sum"]))


Duplikatet: vendimi


# Vendimi për duplikatet

## Evidenca

Analiza e Dafinës: commit `d2708c640de830014edf4c4f35a528849c98455f`, `reports/uniqueness.md`, `reports/01_duplikatet_shqyrtimi.csv` dhe `processed/duplicate_candidates.csv`. Këto lexohen me git show nga versioni i fiksuar, pa i ndryshuar. Nëse mungon commit-i, kërkohet git fetch origin. SHA-256 e kandidatëve: `65632c8f989f1d70caac47d9338127e7518bb747ac749db2fbb99ff7c25e09a3`. Dataset-i burimor i atij commit-i përputhet me origjinalin lokal.

Ka 3,132 rreshta në 1,288 grupe kandidatë. Dafina propozon 378 rreshta të tepërt. Vendimi këtu pranon 235 grupe dhe liston 375 rreshta për fshirjen e ardhshme. Asnjë rresht nuk është fshirë.

## Decision + Rationale

Njësia e analizës është vëzhgimi/grumbullimi, jo numri i publikimeve të tij. Kriteri kandidat është species + eventDate origjinale + koordinata pa rrumbullakim + recordedBy me hapësira të normalizuara dhe casefold. Kandidatët me çelës jo të plotë nuk bashkohen; identitetet alternative të të njëjtit person nuk hamendësohen. Nuk bashkohen timestamp-e të ndryshme vetëm pse bien në të njëjtën ditë.

Pranohen vetëm grupet e propozuara nga Dafina që kanë precizion ditor, vëzhgues të njohur, scientificName dhe basisOfRecord konsistent, pa qendër rrjete 7071 m dhe pa konflikt në individualCount. Intervalet/datat e pjesshme ruhen. Specimenët me catalogNumber të ndryshëm ruhen, përveç grupeve ndërburimore të konfirmuara shprehimisht nga shqyrtimi i Dafinës. Vetëm fakti që datasetKey ndryshon nuk provon dyfishim. Përjashtimet ndaj propozimit të saj dokumentohen grup më grup në vendimet_e_grupeve_duplikate.csv.

Ky është rregull operacional për deduplikim të regjistrimeve të vëzhgimit; barazia e vendit, datës, vëzhguesit dhe numrit nuk provon identitetin biologjik të individëve. Vizitat e përsëritura mund të jenë vëzhgime të vlefshme. Kjo pasiguri ruhet në këtë arsyetim dhe lista është e auditueshme para largimit; nuk pretendohet verifikim manual i çdo grupi.

## Cili rresht ruhet

Ruhet rreshti me numrin më të madh të fushave semantikisht të plotësuara në këtë listë fikse: scientificName, species, kingdom, phylum, class, order, family, genus, taxonRank, eventDate, year, month, day, decimalLatitude, decimalLongitude, coordinateUncertaintyInMeters, elevation, locality, municipality, district, protectedArea, protectedAreaDesignation, strictlyProtected, basisOfRecord, individualCount, recordedBy, identifiedBy, iucnRedListCategory. Mungesat efektive përfshijnë tokenët e konfirmuar dhe elevation=0 për vlerësim; kjo nuk e zëvendëson ende elevation në dataset. ID-të, metadata e botimit, flamujt e rinj dhe iucnRedListStatus redundant nuk japin pikë. Në barazim ruhet gbifID numerik më i vogël, vetëm për qëndrueshmëri, jo sepse është më cilësor. Mungesat e rreshtit të mbajtur nuk mbushen nga rreshta të tjerë në këtë detyrë.

## individualCount

Kur numërimet ndryshojnë, ruhet i gjithë grupi. Një numër i dhënë dhe një mungesë trajtohen gjithashtu si konflikt. Kur numërimet janë të njëjta ruhet numri i rreshtit të përzgjedhur, pa mbledhje, mesatare apo maksimum. Kur të gjitha mungojnë, mbeten NaN. Numërimet e ndryshme mund të jenë vizita të ndryshme: as ruajtja e tyre nuk autorizon mbledhjen e tyre si bollëk biologjik.

## Dokumentimi i detyrës

1. **Input:** kandidatët dhe shqyrtimi i Dafinës nga commit-i i mësipërm, df_raw dhe df_clean.
2. **Çka u bë:** U vendos rregulli, përjashtimet dhe renditja e kandidatëve; u përgatit lista e rreshtave për fshirje.
3. **Rreshta para/pas:** 46,831 / 46,831; **kolona para/pas:** 79 / 79.
4. **Pse u bë:** Për të shmangur fshirjen e ekzemplarëve/numërimeve të ndryshme dhe për të pasur përzgjedhje të auditueshme.
5. **Output:** reports/vendimi_per_duplikatet.md, reports/vendimet_e_grupeve_duplikate.csv, reports/renditja_e_kandidateve_duplikate.csv, processed/rreshtat_per_fshirje_duplikate.csv.
6. **Commit:** Document duplicate decisions and select removal candidates (mesazh i propozuar).

Largimi i rreshtave është detyrë tjetër. cleaning_log.csv nuk merr transformim fshirjeje për këtë vendim, sepse df_clean nuk ndryshon. protectedArea është plotësuar sipas sqarimit semantik të përdorueses; shih vendimet_per_mungesat_strukturore.md për burimin dhe kufizimet.


size   sum
propozimi_Dafina pranohet            
False            False     1050  2516
True             False        3     6
                 True       235   610

### Dokumentimi i hapit: Vendimi për duplikatet

1. **Input:** kandidatët dhe shqyrtimi i Dafinës në commit d2708c640de830014edf4c4f35a528849c98455f.
2. **Çka u bë:** U përcaktuan kriteret, përjashtimet, rreshti për ruajtje dhe lista për fshirje të mëvonshme.
3. **Rreshta para/pas:** 46,831 / 46,831; **kolona para/pas:** 79 / 79.
4. **Pse u bë:** Përzgjedhje e qëndrueshme dhe pa humbje të numërimeve konfliktuale.
5. **Output:** reports/vendimi_per_duplikatet.md, reports/vendimet_e_grupeve_duplikate.csv, reports/renditja_e_kandidateve_duplikate.csv, processed/rreshtat_per_fshirje_duplikate.csv.
6. **Commit:** Document duplicate decisions and select removal candidates (mesazh i propozuar).
7. **Decision + Rationale:** Ruhet rreshti më i plotë; në barazim gbifID më i vogël. Numërimet konfliktuale nuk bashkohen, nuk imputohen dhe nuk mblidhen. Shih raportin për kufizimet dhe përjashtimet.